# 🌊 DAQathon Session 1: Machine Learning

This notebook trains the machine-learning examples for Session 1: Random Forest, k-means, CNN, and Transformer. It starts from a prepared parquet cache, creates the same train / validation / test dataframes used throughout the workflow, and then lets each model section focus on modelling rather than data-preparation plumbing.

The workshop includes pre-loaded datasets and prepared caches, so the default path is fast: choose a dataset preset, choose a data fraction, run the setup cell, and continue to the model sections. If you want to use your own data, the setup area also includes a commented cell to build a simple parquet cache from CSV files and a second commented cell to load any existing custom cache.

Use `session1_data_preparation.ipynb` when you want the slower, more explanatory walkthrough of raw CSV inspection, parquet caching, reviewed-row accounting, and split design. Use this notebook when you are ready to train and compare models.

Notebook outline:

1. **Dataset and cache setup.** Load a pre-loaded workshop cache or configure your own data.
2. **Random Forest.** Train a supervised tabular baseline.
3. **k-means.** Explore unsupervised clustering and compare clusters with known labels.
4. **CNN.** Train a sequence model that sees short windows of sensor history.
5. **Transformer.** Train sequence models for classification and next-value anomaly detection.
6. **Reflection.** Compare model behaviours and identify practical next steps.


## 🧰 One-Time Cluster Setup

Before you use these notebooks on Narval or FIR for the first time, open a cluster terminal and run:

```bash
/project/def-kmoran/shared/daqathon/envs/daqathon-ml-venv/bin/jupyter kernelspec install --user /project/def-kmoran/shared/daqathon/kernels/daqathon-ml
```

This is a **one-time per-user step**. After that:

- refresh JupyterHub if it was already open,
- open the notebook from your cloned repo,
- select the shared `Daqathon ML` kernel.


## Import Packages And Helper Functions

Run the next cell once before choosing a dataset setup option. It prepares the Python environment by importing plotting/data libraries, scikit-learn tools, and the DAQathon helper functions used later in the model sections.

After the imports are ready, choose **Option A** for a prepared workshop cache, **Option B** to create a cache from your own CSV files, or **Option C** to load an existing custom cache.

In [ ]:
# Standard-library imports used throughout the notebook.
from pathlib import Path
import copy
import json
import os
import pickle
import subprocess
import sys
from time import perf_counter

# Find the repo root before importing utilities from scripts/.
# This lets the notebook run from either the repo root or the notebooks/ folder.
NOTEBOOK_ROOT = Path.cwd()
for candidate_root in [NOTEBOOK_ROOT, *NOTEBOOK_ROOT.parents]:
    if (candidate_root / "notebooks").exists() and (candidate_root / "scripts").exists():
        NOTEBOOK_ROOT = candidate_root
        break
if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_ROOT))

# Data, plotting, and machine-learning packages.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from matplotlib.lines import Line2D
from matplotlib.ticker import PercentFormatter
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

# Session setup helpers.
from scripts.session1_resume_utils import load_ml_section_state, show_setup_json

# Shared modelling helpers used by the RF, k-means, CNN, and Transformer sections.
from scripts.session1_modeling import (
    add_temporal_context_features,
    add_tabular_baseline_features,
    apply_target_strategy,
    build_cache_bundle_paths,
    build_labeled_intervals,
    build_model_frame,
    build_reviewed_target_frame,
    build_sequence_split_bundle,
    build_sequence_split_bundle_from_frames,
    build_sequence_label_interval_data,
    build_window_classification_interval_data,
    clean_source_file_label,
    compute_contiguous_split_target_distribution,
    compute_interval_classification_metrics,
    compute_split_share_gap,
    DEFAULT_FLAG_PALETTE,
    evaluate_classifier,
    fit_extra_trees,
    fit_kmeans,
    fit_random_forest,
    infer_interval_origin,
    load_full_row_level_frame,
    load_rows_for_time_range,
    load_selected_row_level_frame,
    materialize_reviewed_split_frames,
    merge_adjacent_intervals,
    plot_cluster_window_examples,
    plot_flag_examples,
    plot_time_series_with_bands,
    predict_cnn_window_model,
    predict_sequence_label_cnn,
    predict_transformer_sequence_model,
    predict_transformer_window_model,
    resolve_cache_bundle_paths,
    resolve_runtime_output_root,
    report_average,
    run_cnn_search,
    run_cnn_search_from_frames,
    run_rf_search,
    sample_frame_by_strategy,
    scan_interleaved_block_rows,
    select_time_range,
    split_frame_by_strategy,
    stage_cache_into_runtime,
    stage_directory_into_runtime,
    summarize_split_distributions,
    summarize_target_by_time_bin,
    SUPPORTED_SPLIT_STRATEGIES,
)

# Notebook display, cache-inspection, split-review, and demo helpers.
from scripts.session1_intro_utils import (
    choose_cache_bundle_paths,
    directory_size_bytes,
    derive_fractional_row_limit,
    evenly_spaced_take,
    filter_csv_paths_with_required_columns,
    load_raw_flag_context_sample,
    load_raw_scalar_sample,
    build_reviewed_modelling_split,
    show_reviewed_modelling_split_build,
    show_reviewed_model_row_accounting,
    show_fixed_split_review,
    show_cnn_interval_demo,
    show_episode_aware_split_comparison,
    summarize_issue_adequacy,
    build_split_strategy_tables,
    show_session1_cache_inspection,
    load_session1_cache_context,
    read_parquet_head,
    select_part_paths,
    stage_session1_inputs,
    show_reviewed_split_summary,
    show_session1_cache_read_comparison,
    show_split_strategy_comparison,
    show_split_strategy_timeline,
    show_temporal_flag_summary,
)

# Generic CSV/parquet cache helpers for participant-owned data.
from scripts.parquet_cache_utils import (
    csv_files_to_row_parquet_cache,
    resolve_csv_paths,
    resolve_or_create_parquet_cache,
)

# ONC-specific cache builders used by the more detailed data-preparation path.
from scripts.onc_scalar_cache_utils import (
    create_onc_row_level_parquet_cache,
    create_onc_window_summary_parquet_cache,
)
from scripts.prepare_scalar_session1_data import (
    DEFAULT_MEASUREMENT_COLUMNS,
    locate_header,
    read_scalar_csv,
)


## Start Here: Dataset And Cache Setup

Run the setup path that matches your data before any model section.

- **Option A** uses a pre-loaded workshop dataset and prepared parquet cache. This is the default path.
- **Option B** creates a parquet cache from your own CSV files. Run **Option C** afterwards to load that new cache.
- **Option C** loads an existing parquet cache, including one you just created with Option B.

The built-in dataset options are `ctd_conductivity`, `fluorometer_turbidity`, `oxygen`, and `conductivity_plugs`. Custom data needs a timestamp column, a target/label column, numeric measurement columns, and good/issue label definitions.

Workshop preset values live in `workshop_config/session1_dataset_profiles.py`. Edit that file if a prepared dataset needs a different target column, feature-column list, plot column, cache stem, or label grouping. Shared label meanings, colours, cache fallbacks, and default measurement-column lists live in `workshop_config/session1_defaults.py`.

If you want the step-by-step explanation of raw CSV inspection, parquet caching, reviewed-row accounting, or split design, work through `session1_data_preparation.ipynb` first. This ML notebook keeps setup concise so you can get to model training quickly.


### Option A: Load A Prepared Workshop Dataset

Use this default path for the pre-loaded Session 1 datasets. The controls below do three things:

- choose the prepared dataset/cache,
- choose how much reviewed data to keep for an interactive run,
- choose the train/validation/test split and the optional train-only subsample.

Most runs only need `DATASET_PROFILE_ID`, `DATA_FRACTION`, `SPLIT_STRATEGY`, and `TRAIN_SUBSET_STRATEGY`. The remaining controls are visible so you can reproduce or override the dataset-building choices from the data-preparation notebook without digging through utility code.

In [ ]:
# Option A: built-in workshop dataset and prepared parquet cache.

# Dataset preset. Options: "ctd_conductivity", "fluorometer_turbidity", "oxygen", or "conductivity_plugs".
DATASET_PROFILE_ID = "ctd_conductivity"

# Optional path/cache overrides. Keep these as None for the workshop shared/local cache autodetection.
READ_RAW_DATA_DIR = None      # Example override: "/path/to/raw_csv_folder".
READ_CACHE_DIR = None         # Example override: "/path/to/prepared/parquet_cache".
CACHE_BUNDLE_NAME = None      # Example override: "my_prepared_cache".

# Dataset-size controls. DATA_FRACTION scales reviewed-row and train-subset caps.
# Use 0.1 for a quick run, 0.5 for a moderate run, 0.9 for a larger run, and 1.0 for no fraction cap.
DATA_FRACTION = 0.1
USE_DATA_FRACTION_BUDGETS = True  # False keeps every reviewed row and removes routine setup caps.

# Reviewed-row loading controls. None means "use the default derived from DATA_FRACTION" where applicable.
REVIEWED_FILE_SELECTION_MODE = "spread"  # Options: "spread", "first", "last", or "all".
REVIEWED_FILE_LIMIT = None                # Limit the number of parquet parts/files loaded, or None for all selected by the profile.
REVIEWED_MODEL_ROW_LIMIT = None           # Override the reviewed modelling row cap, or None to derive it from DATA_FRACTION.

# Train/validation/test split controls.
SPLIT_STRATEGY = "global_contiguous"      # Options: "global_contiguous", "per_source_contiguous", "interleaved_blocks".
INTERLEAVED_BLOCK_ROWS = 1024             # Block size used only when SPLIT_STRATEGY = "interleaved_blocks".
TRAIN_FRACTION = 0.70                     # Fraction of reviewed modelling rows assigned to train.
VALIDATION_FRACTION = 0.15                # Fraction assigned to validation; the rest goes to test.

# Train-only subsampling controls. These affect train_df, not valid_df or test_df.
TRAIN_SUBSET_STRATEGY = "time_spread"     # Options: "time_spread", "issue_focused", "balanced_reviewed", "full_train".
TRAIN_SUBSET_MAX_ROWS = None              # Override max train rows, or None to derive it from DATA_FRACTION.
ISSUE_ROWS_PER_FILE = None                # Used by "issue_focused"; None derives it from DATA_FRACTION.
BALANCED_REVIEWED_TARGET_ISSUE_SHARE = 0.25  # Used by "balanced_reviewed"; 0.25 means target about 25% issue rows.

# Reproducible sampling/training seed.
SEED = 21

# ML_STATE is the setup dictionary used by the rest of this notebook.
# The next setup cell exposes only the core modelling variables/dataframes.
ML_STATE = load_ml_section_state(
    dataset_profile_id=DATASET_PROFILE_ID,
    data_fraction=DATA_FRACTION,
    use_data_fraction_budgets=USE_DATA_FRACTION_BUDGETS,
    split_strategy=SPLIT_STRATEGY,
    train_subset_strategy=TRAIN_SUBSET_STRATEGY,
    reviewed_file_selection_mode=REVIEWED_FILE_SELECTION_MODE,
    reviewed_file_limit=REVIEWED_FILE_LIMIT,
    reviewed_model_row_limit=REVIEWED_MODEL_ROW_LIMIT,
    train_subset_max_rows=TRAIN_SUBSET_MAX_ROWS,
    issue_rows_per_file=ISSUE_ROWS_PER_FILE,
    balanced_reviewed_target_issue_share=BALANCED_REVIEWED_TARGET_ISSUE_SHARE,
    interleaved_block_rows=INTERLEAVED_BLOCK_ROWS,
    train_fraction=TRAIN_FRACTION,
    validation_fraction=VALIDATION_FRACTION,
    seed=SEED,
    read_raw_data_dir=READ_RAW_DATA_DIR,
    read_cache_dir=READ_CACHE_DIR,
    cache_bundle_name=CACHE_BUNDLE_NAME,
    verbose=True,
)


### Option B: Create A Parquet Cache From Your Own CSV Files

Use this cell when your data is still in CSV files. Uncomment the code, edit the paths/columns/labels, and run it to create a row-level parquet cache.

After this finishes, run **Option C** with the same cache folder/name and column settings to load the cache for modelling.


In [ ]:
# # Option B: create a parquet cache from your own CSV files.
# from pathlib import Path
# import sys
#
# # Find the repository root before importing helper functions from scripts/.
# NOTEBOOK_ROOT = Path.cwd()
# for candidate_root in [NOTEBOOK_ROOT, *NOTEBOOK_ROOT.parents]:
#     if (candidate_root / "notebooks").exists() and (candidate_root / "scripts").exists():
#         NOTEBOOK_ROOT = candidate_root
#         break
# if str(NOTEBOOK_ROOT) not in sys.path:
#     sys.path.insert(0, str(NOTEBOOK_ROOT))
#
# from scripts.parquet_cache_utils import resolve_or_create_parquet_cache
#
# # CSV inputs. Use CUSTOM_RAW_CSV_DIR for every *.csv in a folder, or use
# # CUSTOM_RAW_CSV_PATHS for an exact list of files. Leave the other one as None.
# CUSTOM_RAW_CSV_DIR = Path("/path/to/csv_folder")
# CUSTOM_RAW_CSV_PATHS = None  # Example: [Path("/path/to/file1.csv"), Path("/path/to/file2.csv")]
#
# # Parquet output. CUSTOM_CACHE_BUNDLE_NAME becomes the filename stem, for example
# # my_sensor_cache_metadata.json and my_sensor_cache_row_level/.
# CUSTOM_PARQUET_CACHE_DIR = Path("/path/to/parquet_cache_folder")
# CUSTOM_CACHE_BUNDLE_NAME = "my_sensor_cache"
#
# # CSV-reading controls.
# CUSTOM_TIME_COLUMN = "Time UTC"  # Timestamp column; change this if your CSV uses another name, such as "timestamp".
# CUSTOM_CSV_HEADER = "first_row"  # Options: "first_row", "auto" for ONC-style metadata headers, or an integer row number such as 5.
# CUSTOM_TARGET_FLAG = "label"  # Label column the supervised models should predict.
# CUSTOM_MEASUREMENT_COLUMNS = ["sensor_value"]  # Numeric sensor columns used as ML features.
# CUSTOM_OPTIONAL_QC_COLUMNS = []  # Extra context columns to preserve in parquet, or [] if none.
# CUSTOM_ISSUE_LABELS = [1]  # Labels counted as issue/anomaly rows in cache summaries.
#
# # Build the cache only if it is missing. Set force_rebuild_cache=True when you
# # intentionally want to overwrite an existing cache with the same name.
# cache_build_result = resolve_or_create_parquet_cache(
#     cache_root=CUSTOM_PARQUET_CACHE_DIR,
#     cache_bundle_name=CUSTOM_CACHE_BUNDLE_NAME,
#     raw_data_dir=CUSTOM_RAW_CSV_DIR,
#     raw_csv_paths=CUSTOM_RAW_CSV_PATHS,
#     target_flag=CUSTOM_TARGET_FLAG,
#     measurement_columns=CUSTOM_MEASUREMENT_COLUMNS,
#     optional_qc_columns=CUSTOM_OPTIONAL_QC_COLUMNS,
#     issue_labels=CUSTOM_ISSUE_LABELS,
#     time_column=CUSTOM_TIME_COLUMN,
#     csv_header=CUSTOM_CSV_HEADER,
#     build_cache_if_missing=True,
#     force_rebuild_cache=False,
#     generic_csv_cache=True,
# )
# print(cache_build_result["summary"])
#
# # Next step: run Option C below to load this cache into train/validation/test dataframes.


### Option C: Load An Existing Custom Parquet Cache

Use this cell when you already have a parquet cache. That cache might come from Option B above, the data-preparation notebook, or another run.

Uncomment the code, edit the cache path/name and column settings, then run this cell before continuing to the ML sections.


In [ ]:
# # Option C: load an existing parquet cache, then continue with the ML sections.
# from pathlib import Path
# import sys
#
# # Find the repository root before importing helper functions from scripts/.
# NOTEBOOK_ROOT = Path.cwd()
# for candidate_root in [NOTEBOOK_ROOT, *NOTEBOOK_ROOT.parents]:
#     if (candidate_root / "notebooks").exists() and (candidate_root / "scripts").exists():
#         NOTEBOOK_ROOT = candidate_root
#         break
# if str(NOTEBOOK_ROOT) not in sys.path:
#     sys.path.insert(0, str(NOTEBOOK_ROOT))
#
# from scripts.session1_resume_utils import load_ml_section_state
#
# # Cache location. These should match the values used when the cache was created.
# CUSTOM_PARQUET_CACHE_DIR = Path("/path/to/cache_folder")
# CUSTOM_CACHE_BUNDLE_NAME = "my_sensor_cache"
#
# # Dataset-size controls used after the parquet cache is loaded.
# DATA_FRACTION = 0.1  # 0.1 is quick, 0.5 is moderate, 0.9 is larger, and 1.0 removes row caps.
# USE_DATA_FRACTION_BUDGETS = True  # False keeps every reviewed row and removes routine setup caps.
# REVIEWED_FILE_SELECTION_MODE = "spread"  # Options: "spread", "first", "last", or "all".
# REVIEWED_FILE_LIMIT = None  # Limit parquet parts/files, or None for all selected by the cache/profile.
# REVIEWED_MODEL_ROW_LIMIT = None  # Override reviewed modelling row cap, or None to derive from DATA_FRACTION.
#
# # Split controls.
# SPLIT_STRATEGY = "global_contiguous"  # Options: "global_contiguous", "per_source_contiguous", "interleaved_blocks".
# INTERLEAVED_BLOCK_ROWS = 1024  # Used only when SPLIT_STRATEGY = "interleaved_blocks".
# TRAIN_FRACTION = 0.70
# VALIDATION_FRACTION = 0.15
#
# # Train-only subsampling controls. These affect train_df, not valid_df or test_df.
# TRAIN_SUBSET_STRATEGY = "time_spread"  # Options: "time_spread", "issue_focused", "balanced_reviewed", "full_train".
# TRAIN_SUBSET_MAX_ROWS = None  # Override max train rows, or None to derive from DATA_FRACTION.
# ISSUE_ROWS_PER_FILE = None  # Used by "issue_focused"; None derives it from DATA_FRACTION.
# BALANCED_REVIEWED_TARGET_ISSUE_SHARE = 0.25  # Used by "balanced_reviewed"; 0.25 means target about 25% issue rows.
# SEED = 21  # Reproducible sampling/training seed.
#
# # Column and label settings. These tell the ML sections what your parquet rows mean.
# CUSTOM_TARGET_FLAG = "label"  # Label column the supervised models should predict.
# CUSTOM_MEASUREMENT_COLUMNS = ["sensor_value"]  # Numeric sensor columns used by RF/CNN/Transformer/k-means.
# CUSTOM_OPTIONAL_QC_COLUMNS = []  # Extra context columns to load, or [] if none.
# CUSTOM_GOOD_LABELS = [0]  # Labels treated as normal/good examples.
# CUSTOM_ISSUE_LABELS = [1]  # Labels treated as issue/anomaly examples.
#
# # Plot columns. Usually choose one primary sensor column and, optionally, one context column.
# CUSTOM_PLOT_MEASUREMENT_COLUMN = "sensor_value"
# CUSTOM_PLOT_SECONDARY_COLUMN = None
#
# # This creates ML_STATE from your custom cache. Run the next active cell afterwards
# # so the Random Forest, k-means, CNN, and Transformer sections receive the same variables.
# ML_STATE = load_ml_section_state(
#     dataset_profile_id="custom",  # Custom template; explicit settings below define your dataset.
#     data_fraction=DATA_FRACTION,
#     use_data_fraction_budgets=USE_DATA_FRACTION_BUDGETS,
#     split_strategy=SPLIT_STRATEGY,
#     train_subset_strategy=TRAIN_SUBSET_STRATEGY,
#     reviewed_file_selection_mode=REVIEWED_FILE_SELECTION_MODE,
#     reviewed_file_limit=REVIEWED_FILE_LIMIT,
#     reviewed_model_row_limit=REVIEWED_MODEL_ROW_LIMIT,
#     train_subset_max_rows=TRAIN_SUBSET_MAX_ROWS,
#     issue_rows_per_file=ISSUE_ROWS_PER_FILE,
#     balanced_reviewed_target_issue_share=BALANCED_REVIEWED_TARGET_ISSUE_SHARE,
#     interleaved_block_rows=INTERLEAVED_BLOCK_ROWS,
#     train_fraction=TRAIN_FRACTION,
#     validation_fraction=VALIDATION_FRACTION,
#     seed=SEED,
#     parquet_cache_dir=CUSTOM_PARQUET_CACHE_DIR,
#     cache_bundle_name=CUSTOM_CACHE_BUNDLE_NAME,
#     target_flag=CUSTOM_TARGET_FLAG,
#     measurement_columns=CUSTOM_MEASUREMENT_COLUMNS,
#     optional_qc_columns=CUSTOM_OPTIONAL_QC_COLUMNS,
#     good_labels=CUSTOM_GOOD_LABELS,
#     issue_labels=CUSTOM_ISSUE_LABELS,
#     plot_measurement_column=CUSTOM_PLOT_MEASUREMENT_COLUMN,
#     plot_secondary_column=CUSTOM_PLOT_SECONDARY_COLUMN,
#     kmeans_feature_mode="row_level",  # Custom CSV caches usually do not include window-summary parquet.
#     verbose=True,
# )


### Review The Loaded Dataset Choices

`ML_STATE` now holds the selected cache paths, label settings, reviewed modelling dataframe, train/validation/test split, and train-only subset. Later model sections read those values directly from `ML_STATE`, so there is one resolved setup dictionary instead of a second layer of notebook-wide copies.

Setup-time choices are applied when Option A or Option C creates `ML_STATE`. To change them, edit the controls in that option and rerun its load cell.

| Setting | What it controls | How to change it |
|---|---|---|
| `DATA_FRACTION` | Overall setup size knob for reviewed rows and routine train-subset caps | Edit `DATA_FRACTION` in Option A or Option C |
| `REVIEWED_FILE_SELECTION_MODE` / `REVIEWED_FILE_LIMIT` | Which parquet parts/files are loaded before reviewed rows are counted | Edit those controls; keep `limit=None` to use all selected parts |
| `REVIEWED_MODEL_ROW_LIMIT` | Maximum reviewed rows kept before split comparison | Leave `None` to derive from `DATA_FRACTION`, or set an integer row cap |
| `SPLIT_STRATEGY` | How reviewed rows become train/validation/test | Choose `"global_contiguous"`, `"per_source_contiguous"`, or `"interleaved_blocks"` |
| `INTERLEAVED_BLOCK_ROWS` | Block size for the interleaved split strategy | Edit when `SPLIT_STRATEGY = "interleaved_blocks"` |
| `TRAIN_SUBSET_MAX_ROWS` | Maximum training rows kept by the train-only subsampling step | Leave `None` to derive from `DATA_FRACTION`, or set an integer row cap |
| `ISSUE_ROWS_PER_FILE` | Extra issue rows requested by the `issue_focused` train-subset strategy | Leave `None` to derive from `DATA_FRACTION`, or set an integer |
| `BALANCED_REVIEWED_TARGET_ISSUE_SHARE` | Target issue-row share for the `balanced_reviewed` train-subset strategy | Edit this fraction; `0.25` means target about 25% issue rows |

Model-specific caps are defined at the start of each model section, right before they are used. For example, the Random Forest section defines its own fit/evaluation row caps, and the k-means section defines its own window/row cap.

## Part 1 — Random Forest


### 🌲 Supervised Learning: Random Forest

A **Random Forest** is an ensemble of many decision trees.

Here is the basic idea:

1. make many slightly different training samples from the data,
2. train one decision tree on each sample,
3. let the trees vote on the final class.

Each tree asks simple if/then questions such as:

- is conductivity above some threshold?
- did temperature change sharply?
- is the measurement happening at a particular time of day?

In this notebook, the Random Forest trains on a tabular feature vector built from each reviewed row.
Right before fitting the forest, we create features that include:

- the selected measurement columns themselves,
- `abs_delta` features that measure how sharply each measurement changed from the previous row,
- and simple clock features such as hour, minute, and day of year.

That feature-building step is specific to the Random Forest baseline. The CNN and transformer sections later do **not** use this same tabular feature table.

Why this is a good Session 1 model:

- it works well on numeric tabular data,
- it usually needs less feature scaling ceremony than many other models,
- it gives us interpretable outputs such as feature importance.

Limits to keep in mind:

- it does not naturally understand long sequential context,
- it only learns from the features we explicitly give it,
- rare classes can still be difficult.


![Random Forest bagging illustration](../assets/session1/random_forest_bagging_illustration.png)

Diagram idea: bootstrap samples are drawn from the original dataset, separate trees are trained, and their predictions are aggregated into one model decision.

Image credit: Harry585, Wikimedia Commons, CC BY-SA 4.0. Source: [File:Random Forest Bagging Illustration.png](https://commons.wikimedia.org/wiki/File:Random_Forest_Bagging_Illustration.png)


### Random Forest Settings

These settings control the baseline forest we train below.

Main variables:

- `imputer_strategy`: how missing numeric feature values are filled before fitting. Common choices are `"median"`, `"mean"`, `"most_frequent"`, and `"constant"`; `"median"` is a robust default when sensor values have spikes or skew.
- `n_estimators`: number of trees in the forest.
- `max_depth`: maximum depth of each tree. `None` means grow until other stopping rules apply.
- `min_samples_leaf`: minimum number of samples allowed in a leaf.
- `min_samples_split`: minimum number of samples needed to split an internal node.
- `max_features`: how many features each split is allowed to consider. Common choices are `"sqrt"`, `"log2"`, or `None`.
- `class_weight`: how strongly to compensate for imbalance. Common choices are `None`, `"balanced"`, and `"balanced_subsample"`.
- `trees_per_step`: how many trees to add each time we print progress.

Forests do **not** train epoch-by-epoch like neural networks. To make progress visible, we grow the forest in chunks using `warm_start=True` and print the validation F1 after each chunk.

One practical note: when we use `warm_start=True`, this notebook converts `"balanced"` or `"balanced_subsample"` into one fixed class-weight dictionary computed from `y_train`. That avoids a scikit-learn warning and keeps the weighting consistent across growth steps.


In [ ]:
RF_CONFIG = {
    # Fill missing numeric feature values before fitting the forest.
    # Options: "median" (robust default), "mean", "most_frequent", or "constant".
    "imputer_strategy": "median",
    "n_estimators": 200,
    "trees_per_step": 25,
    "max_depth": None,
    "min_samples_leaf": 2,
    "min_samples_split": 2,
    "max_features": "sqrt",
    "class_weight": "balanced_subsample",
    "verbose": 0,
}

display(pd.Series(RF_CONFIG, name="value").rename_axis("rf_parameter").to_frame())


#### Random Forest Row Budget

The Random Forest uses the same `DATA_FRACTION` budget as the rest of the notebook. The next cell applies the derived caps and prints the actual row counts used for train, validation, and test.


In [ ]:
# Random Forest row caps are defined here because they only affect this section.
# Set any *_MAX_ROWS value to None after this block if you want RF to use every row in that split.
# rows_limit is the helper argument meaning "keep at most this many rows".
RF_TRAIN_BASE_ROWS = 250_000       # Training rows used when DATA_FRACTION = 1.0 and caps are on.
RF_TRAIN_MIN_ROWS = 25_000         # Smallest training cap after scaling by DATA_FRACTION.
RF_EVAL_BASE_ROWS = 150_000        # Validation/test rows used when DATA_FRACTION = 1.0 and caps are on.
RF_EVAL_MIN_ROWS = 15_000          # Smallest validation/test cap after scaling.

if ML_STATE['USE_DATA_FRACTION_BUDGETS'] and ML_STATE['DATA_FRACTION'] < 0.999:
    RF_TRAIN_MAX_ROWS = max(RF_TRAIN_MIN_ROWS, int(RF_TRAIN_BASE_ROWS * ML_STATE['DATA_FRACTION']))
    RF_VALIDATION_MAX_ROWS = max(RF_EVAL_MIN_ROWS, int(RF_EVAL_BASE_ROWS * ML_STATE['DATA_FRACTION']))
    RF_TEST_MAX_ROWS = max(RF_EVAL_MIN_ROWS, int(RF_EVAL_BASE_ROWS * ML_STATE['DATA_FRACTION']))
else:
    RF_TRAIN_MAX_ROWS = None
    RF_VALIDATION_MAX_ROWS = None
    RF_TEST_MAX_ROWS = None

# Example overrides:
# RF_TRAIN_MAX_ROWS = None
# RF_VALIDATION_MAX_ROWS = 50_000
# RF_TEST_MAX_ROWS = 50_000

# If train_df was already subsetted, use the same strategy when we cap it again.
# If the active training strategy is "full_train", use "time_spread" so a cap
# keeps rows across the full training time span instead of only the earliest rows.
RF_TRAIN_CAP_STRATEGY = (
    ML_STATE['TRAIN_SUBSET_STRATEGY']
    if ML_STATE['TRAIN_SUBSET_STRATEGY'] != "full_train"
    else "time_spread"
)

rf_train_fit_df = (
    sample_frame_by_strategy(
        ML_STATE['train_df'],
        rows_limit=RF_TRAIN_MAX_ROWS,
        sample_strategy=RF_TRAIN_CAP_STRATEGY,
        target_flag=ML_STATE['TARGET_FLAG'],
        good_labels=ML_STATE['GOOD_LABELS'],
        issue_labels=ML_STATE['ISSUE_LABELS'],
        issue_rows=ML_STATE['ISSUE_ROWS_PER_FILE'],
        balanced_issue_share=ML_STATE['BALANCED_REVIEWED_TARGET_ISSUE_SHARE'],
    )
    if RF_TRAIN_MAX_ROWS is not None and len(ML_STATE['train_df']) > RF_TRAIN_MAX_ROWS
    else ML_STATE['train_df'].copy()
)
# Validation/test use time-spread caps because they should stay representative,
# not issue-enriched. They are still held-out rows.
rf_valid_eval_df = (
    evenly_spaced_take(ML_STATE['valid_df'].sort_values("Time UTC").reset_index(drop=True), RF_VALIDATION_MAX_ROWS)
    if RF_VALIDATION_MAX_ROWS is not None and len(ML_STATE['valid_df']) > RF_VALIDATION_MAX_ROWS
    else ML_STATE['valid_df'].copy()
)
rf_test_eval_df = (
    evenly_spaced_take(ML_STATE['test_df'].sort_values("Time UTC").reset_index(drop=True), RF_TEST_MAX_ROWS)
    if RF_TEST_MAX_ROWS is not None and len(ML_STATE['test_df']) > RF_TEST_MAX_ROWS
    else ML_STATE['test_df'].copy()
)

show_setup_json(
    {
        "DATA_FRACTION": ML_STATE['DATA_FRACTION'],
        "USE_DATA_FRACTION_BUDGETS": ML_STATE['USE_DATA_FRACTION_BUDGETS'],
        "RF_TRAIN_MAX_ROWS": RF_TRAIN_MAX_ROWS,
        "RF_VALIDATION_MAX_ROWS": RF_VALIDATION_MAX_ROWS,
        "RF_TEST_MAX_ROWS": RF_TEST_MAX_ROWS,
        "rf_train_cap_strategy": RF_TRAIN_CAP_STRATEGY,
        "rf_train_rows_used": len(rf_train_fit_df),
        "rf_validation_rows_used": len(rf_valid_eval_df),
        "rf_test_rows_used": len(rf_test_eval_df),
    }
)


### Fit The Random Forest

The row-budget cell chose which labelled rows will be used for fitting and evaluation. The next cell reloads those selected rows from parquet, builds the tabular feature table, fits the imputer, trains the Random Forest, and saves the fitted pipeline for later cells.


In [ ]:
import gc

rf_split_rows = materialize_reviewed_split_frames(
    ML_STATE['selected_paths'],
    {
        "train": rf_train_fit_df,
        "validation": rf_valid_eval_df,
        "test": rf_test_eval_df,
    },
    columns=ML_STATE['ROW_USE_COLUMNS'],
    target_flag=ML_STATE['TARGET_FLAG'],
    task_mode=ML_STATE['task_mode'],
    good_labels=ML_STATE['GOOD_LABELS'],
    issue_labels=ML_STATE['ISSUE_LABELS'],
)
rf_train_rows_df = rf_split_rows["train"]
rf_valid_rows_df = rf_split_rows["validation"]
rf_test_rows_df = rf_split_rows["test"]

# Build the Random Forest feature table here. This is the step where the shared
# helper adds the tabular baseline features:
# - the selected measurement columns,
# - per-column `abs_delta` change features,
# - and simple UTC clock features.
rf_train_fit_df, feature_columns, _ = build_model_frame(
    rf_train_rows_df,
    target_flag=ML_STATE['TARGET_FLAG'],
    measurement_columns=ML_STATE['measurement_columns'],
    task_mode=ML_STATE['task_mode'],
    good_labels=ML_STATE['GOOD_LABELS'],
    issue_labels=ML_STATE['ISSUE_LABELS'],
    model_row_limit=None,
)
rf_valid_eval_df, _, _ = build_model_frame(
    rf_valid_rows_df,
    target_flag=ML_STATE['TARGET_FLAG'],
    measurement_columns=ML_STATE['measurement_columns'],
    task_mode=ML_STATE['task_mode'],
    good_labels=ML_STATE['GOOD_LABELS'],
    issue_labels=ML_STATE['ISSUE_LABELS'],
    model_row_limit=None,
)
rf_test_eval_df, _, _ = build_model_frame(
    rf_test_rows_df,
    target_flag=ML_STATE['TARGET_FLAG'],
    measurement_columns=ML_STATE['measurement_columns'],
    task_mode=ML_STATE['task_mode'],
    good_labels=ML_STATE['GOOD_LABELS'],
    issue_labels=ML_STATE['ISSUE_LABELS'],
    model_row_limit=None,
)

# The raw materialized split frames are no longer needed once the feature tables exist.
# Releasing them keeps high-fraction runs from carrying duplicate copies of the data.
for _rf_name in ["rf_split_rows", "rf_train_rows_df", "rf_valid_rows_df", "rf_test_rows_df"]:
    globals().pop(_rf_name, None)
gc.collect()

# Step 1: fit the imputer once on training data and reuse it for validation/test.
rf_imputer = SimpleImputer(strategy=RF_CONFIG.get("imputer_strategy", "median"))
X_train_rf = rf_imputer.fit_transform(rf_train_fit_df[feature_columns]).astype(np.float32, copy=False)
X_valid_rf = rf_imputer.transform(rf_valid_eval_df[feature_columns]).astype(np.float32, copy=False)
y_train_rf = rf_train_fit_df["model_target"].reset_index(drop=True)
y_valid_rf = rf_valid_eval_df["model_target"].reset_index(drop=True)
y_test_rf = rf_test_eval_df["model_target"].reset_index(drop=True)

# Step 2: compute one stable class-weight dictionary for the whole training split.
requested_class_weight = RF_CONFIG.get("class_weight")
if requested_class_weight in {"balanced", "balanced_subsample"}:
    rf_classes = np.array(sorted(pd.Series(y_train_rf).dropna().unique()))
    rf_weight_values = compute_class_weight(
        class_weight="balanced",
        classes=rf_classes,
        y=y_train_rf,
    )
    effective_class_weight = {
        int(label) if isinstance(label, (np.integer, int)) else label: float(weight)
        for label, weight in zip(rf_classes.tolist(), rf_weight_values.tolist())
    }
else:
    effective_class_weight = requested_class_weight

print(
    {
        "requested_class_weight": requested_class_weight,
        "effective_class_weight": effective_class_weight,
    }
)

# Step 3: build the forest itself. We use warm_start so we can grow it in chunks and print progress.
rf_model = RandomForestClassifier(
    n_estimators=RF_CONFIG["trees_per_step"],
    warm_start=True,
    max_depth=RF_CONFIG["max_depth"],
    min_samples_leaf=RF_CONFIG["min_samples_leaf"],
    min_samples_split=RF_CONFIG["min_samples_split"],
    max_features=RF_CONFIG["max_features"],
    class_weight=effective_class_weight,
    n_jobs=-1,
    random_state=ML_STATE['SEED'],
    verbose=RF_CONFIG["verbose"],
)

rf_progress_rows = []
total_trees = RF_CONFIG["n_estimators"]
trees_per_step = RF_CONFIG["trees_per_step"]
growth_schedule = list(range(trees_per_step, total_trees + 1, trees_per_step))
if not growth_schedule:
    growth_schedule = [total_trees]
elif growth_schedule[-1] != total_trees:
    growth_schedule.append(total_trees)

for tree_count in growth_schedule:
    rf_model.set_params(n_estimators=tree_count)
    rf_model.fit(X_train_rf, y_train_rf)
    current_valid_predictions = rf_model.predict(X_valid_rf)
    current_valid_f1 = f1_score(
        y_valid_rf,
        current_valid_predictions,
        average=report_average(ML_STATE['task_mode']),
        zero_division=0,
    )
    rf_progress_rows.append({"trees_built": tree_count, "validation_f1": float(current_valid_f1)})
    print(f"Built {tree_count:>3} trees | validation F1 = {current_valid_f1:.4f}")

# Step 4: wrap the fitted pieces into a reusable sklearn pipeline object.
rf_pipeline = Pipeline(
    steps=[
        ("imputer", rf_imputer),
        ("model", rf_model),
    ]
)

X_test_rf = rf_imputer.transform(rf_test_eval_df[feature_columns]).astype(np.float32, copy=False)
valid_predictions = rf_model.predict(X_valid_rf)
test_predictions = rf_model.predict(X_test_rf)
labels = sorted(pd.unique(pd.concat([y_train_rf, y_valid_rf, y_test_rf])))

display(pd.DataFrame(rf_progress_rows))

with ML_STATE["RF_MODEL_PATH"].open("wb") as handle:
    pickle.dump(
        {
            "pipeline": rf_pipeline,
            "feature_columns": feature_columns,
            "labels": labels,
            "task_mode": ML_STATE['task_mode'],
        },
        handle,
    )

print(
    {
        "saved_random_forest_model": str(ML_STATE["RF_MODEL_PATH"]),
        "rf_train_rows_used": int(len(rf_train_fit_df)),
        "rf_validation_rows_used": int(len(rf_valid_eval_df)),
        "rf_test_rows_used": int(len(rf_test_eval_df)),
    }
)

# The fitted pipeline keeps the trained model and imputer. The arrays and
# temporary fit frames below are only needed while fitting and predicting.
for _rf_name in [
    "X_train_rf",
    "X_valid_rf",
    "X_test_rf",
    "current_valid_predictions",
    "rf_train_fit_df",
    "rf_valid_eval_df",
    "rf_model",
    "rf_imputer",
    "y_train_rf",
]:
    globals().pop(_rf_name, None)
gc.collect()


### Evaluate Validation And Test Performance

Now that the forest is fitted, we score it on validation and test rows separately. Use the validation results for tuning decisions, and treat the test results as the held-out check of how the model generalises.


In [ ]:
# Report validation and test metrics separately for the RF evaluation slices.
metric_rows = []
for split_name, y_true, y_pred in [
    ("validation", y_valid_rf, valid_predictions),
    ("test", y_test_rf, test_predictions),
]:
    split_f1 = f1_score(y_true, y_pred, average=report_average(ML_STATE['task_mode']), zero_division=0)
    metric_rows.append({"split": split_name, "f1": round(float(split_f1), 4)})
    print(f"{split_name.title()} macro/binary F1: {split_f1:.4f}")
    print(classification_report(y_true, y_pred, labels=labels, zero_division=0))

display(pd.DataFrame(metric_rows))

# Plot normalised confusion matrices for both validation and test.
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
ConfusionMatrixDisplay.from_predictions(
    y_valid_rf,
    valid_predictions,
    labels=labels,
    display_labels=[str(label) for label in labels],
    normalize="true",
    cmap="Blues",
    values_format=".2f",
    xticks_rotation=45,
    ax=axes[0],
)
axes[0].set_title(f"Validation confusion matrix for {ML_STATE['target_name']}")

ConfusionMatrixDisplay.from_predictions(
    y_test_rf,
    test_predictions,
    labels=labels,
    display_labels=[str(label) for label in labels],
    normalize="true",
    cmap="Blues",
    values_format=".2f",
    xticks_rotation=45,
    ax=axes[1],
)
axes[1].set_title(f"Test confusion matrix for {ML_STATE['target_name']}")
fig.suptitle("How to read these plots: each row is a true class, and darker diagonal cells are better.", y=1.03)
plt.tight_layout()
plt.show()


### Interpret The Random Forest

Aggregate metrics say how well the model performed; the next cell asks what it relied on. It shows the most important features and a small sample of mistakes so the numbers are easier to connect back to real sensor records.


In [ ]:
# Feature importance tells us which columns the forest used most strongly.
feature_importances = pd.Series(
    rf_pipeline.named_steps["model"].feature_importances_,
    index=feature_columns,
).sort_values(ascending=False)

top_importances = feature_importances.head(12).sort_values()
top_importances.plot(kind="barh", figsize=(8, 6), title="Top Random Forest feature importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()
plt.close()
display(feature_importances.head(12).rename("importance").to_frame())

# Show a few test-set mistakes to ground the discussion in real timestamps.
error_preview_columns = [
    "Time UTC",
    ML_STATE['TARGET_FLAG'],
    "issue",
    ML_STATE['PLOT_MEASUREMENT_COLUMN'],
    ML_STATE['PLOT_SECONDARY_COLUMN'],
]
error_preview_columns.extend(
    [
        column
        for column in ML_STATE['MEASUREMENT_COLUMNS']
        if column not in error_preview_columns
    ][:1]
)
error_preview_columns = [column for column in error_preview_columns if column in rf_test_eval_df.columns]
error_frame = rf_test_eval_df[error_preview_columns].copy()
error_frame["prediction"] = test_predictions
error_frame["correct"] = error_frame["prediction"] == y_test_rf.to_numpy()
error_examples = error_frame.loc[~error_frame["correct"]].head(12)
print({"test_errors_shown": len(error_examples), "test_error_rate": round(float((~error_frame["correct"]).mean()), 4)})
display(error_examples)

# The test prediction arrays are no longer needed after the mistake preview.
for _rf_name in [
    "rf_test_eval_df",
    "y_test_rf",
    "test_predictions",
    "error_frame",
    "error_examples",
    "feature_importances",
    "top_importances",
]:
    globals().pop(_rf_name, None)
gc.collect()


### Date-Range Demo: Predict QC Flags with the Random Forest

The test-set metrics above summarise performance across the whole held-out split. This mini-workflow asks a more concrete question:

**What does the Random Forest predict on one specific interval of time?**

Use UTC strings such as `"2025-09-10 00:00:00Z"` if you want to override the default range.


In [ ]:
RF_RANGE_START = None
RF_RANGE_END = None
RF_AUTO_SELECT_TEST_RANGE = True
RF_MAX_POINTS_TO_PLOT = 800

print(
    {
        "RF_RANGE_START": RF_RANGE_START,
        "RF_RANGE_END": RF_RANGE_END,
        "RF_AUTO_SELECT_TEST_RANGE": RF_AUTO_SELECT_TEST_RANGE,
        "RF_MAX_POINTS_TO_PLOT": RF_MAX_POINTS_TO_PLOT,
    }
)


The previous cell chooses a manual UTC range or asks the notebook to auto-select a held-out range. The next cell reloads only that interval, applies the fitted forest, and plots true versus predicted labels through time.


In [ ]:
import gc

rf_range_selection = select_time_range(
    ML_STATE['test_df'],
    time_column="Time UTC",
    label_column=ML_STATE['TARGET_FLAG'],
    start=RF_RANGE_START,
    end=RF_RANGE_END,
    auto_select=RF_AUTO_SELECT_TEST_RANGE,
    max_points=RF_MAX_POINTS_TO_PLOT,
)

rf_interval_rows = load_rows_for_time_range(
    ML_STATE['metadata'],
    row_cache_dir=Path(ML_STATE["ROW_CACHE_DIR"]),
    start=rf_range_selection["start"],
    end=rf_range_selection["end"],
    columns=ML_STATE['ROW_USE_COLUMNS'],
)

if rf_interval_rows.empty:
    print("No row-level data was found in the requested Random Forest range.")
else:
    rf_interval_model_df, _, _ = build_model_frame(
        rf_interval_rows,
        target_flag=ML_STATE['TARGET_FLAG'],
        measurement_columns=ML_STATE['measurement_columns'],
        task_mode=ML_STATE['task_mode'],
        model_row_limit=None,
    )
    rf_interval_model_df = rf_interval_model_df[
        (rf_interval_model_df["Time UTC"] >= rf_range_selection["start"])
        & (rf_interval_model_df["Time UTC"] <= rf_range_selection["end"])
    ].reset_index(drop=True)

    if rf_interval_model_df.empty:
        print("The selected Random Forest range did not contain usable labelled rows after preparation.")
    else:
        rf_interval_predictions = rf_pipeline.predict(rf_interval_model_df[feature_columns])
        rf_interval_origin = infer_interval_origin(
            rf_range_selection["start"],
            rf_range_selection["end"],
            {"train": ML_STATE['train_df'], "validation": ML_STATE['valid_df'], "test": ML_STATE['test_df']},
        )
        rf_interval_metrics = compute_interval_classification_metrics(
            rf_interval_model_df["model_target"],
            rf_interval_predictions,
            labels=labels,
            average=report_average(ML_STATE['task_mode']),
            target_names=[str(label) for label in labels],
        )
        rf_plot_palette = DEFAULT_FLAG_PALETTE if ML_STATE['task_mode'] == "multiclass" else {0: "#1f77b4", 1: "#d62728"}
        rf_true_intervals = build_labeled_intervals(
            rf_interval_model_df,
            time_column="Time UTC",
            label_column="model_target",
        )
        rf_pred_frame = rf_interval_model_df[["Time UTC"]].copy()
        rf_pred_frame["predicted_label"] = rf_interval_predictions
        rf_pred_intervals = build_labeled_intervals(
            rf_pred_frame,
            time_column="Time UTC",
            label_column="predicted_label",
        )

        print(
            {
                "selection_mode": rf_range_selection["selection_mode"],
                "selected_priority_flag": rf_range_selection["selected_label"],
                "interval_origin": rf_interval_origin,
                "range_start": rf_range_selection["start"].isoformat(),
                "range_end": rf_range_selection["end"].isoformat(),
                "rows_in_interval": int(len(rf_interval_model_df)),
                "interval_f1": rf_interval_metrics["f1"],
            }
        )
        print(rf_interval_metrics["report_text"])

        display(
            pd.DataFrame(
                {
                    "true_count": pd.Series(rf_interval_model_df["model_target"]).value_counts().sort_index(),
                    "predicted_count": pd.Series(rf_interval_predictions).value_counts().sort_index(),
                }
            ).fillna(0).astype(int)
        )

        rf_demo_figure = plot_time_series_with_bands(
            rf_interval_model_df,
            band_specs=[
                {"title": f"True {ML_STATE['FLAG_EXAMPLE_DISPLAY_NAME']}", "intervals": rf_true_intervals, "palette": rf_plot_palette},
                {"title": f"RF {ML_STATE['FLAG_EXAMPLE_DISPLAY_NAME']} predictions", "intervals": rf_pred_intervals, "palette": rf_plot_palette},
            ],
            measurement_column=ML_STATE['PLOT_MEASUREMENT_COLUMN'],
            secondary_column=ML_STATE['PLOT_SECONDARY_COLUMN'],
            max_points=RF_MAX_POINTS_TO_PLOT,
            title="Random Forest predictions on a selected time range",
            label_meanings=ML_STATE['FLAG_EXAMPLE_LABEL_MEANINGS'],
            legend_title=ML_STATE['FLAG_EXAMPLE_DISPLAY_NAME'],
        )
        plt.show()
        plt.close(rf_demo_figure)

        rf_cm_fig, rf_cm_ax = plt.subplots(figsize=(5, 4))
        ConfusionMatrixDisplay(
            confusion_matrix=rf_interval_metrics["confusion_matrix"],
            display_labels=rf_interval_metrics["display_labels"],
        ).plot(ax=rf_cm_ax, cmap="Blues", colorbar=False)
        rf_cm_ax.set_title("Random Forest confusion matrix on the selected range")
        plt.tight_layout()
        plt.show()
        plt.close(rf_cm_fig)

# Random Forest interval intermediates can be rebuilt by rerunning this cell.
for _rf_name in [
    "rf_interval_rows",
    "rf_interval_model_df",
    "rf_interval_predictions",
    "rf_interval_metrics",
    "rf_true_intervals",
    "rf_pred_frame",
    "rf_pred_intervals",
]:
    globals().pop(_rf_name, None)
gc.collect()


### Reflection Questions: Random Forest

1. Which input features seem most responsible for the mistakes in this interval: raw measurements, change features, or time-of-day features?
2. Do the errors line up with the class imbalance we saw earlier, or do they suggest a different modelling problem?
3. If you wanted to improve this interval specifically, would you change the features, the target, or the forest settings first?

Add your own notes or follow-up questions here:

- 
- 
- 


## Part 2 — k-means


### 🧭 Unsupervised Learning: k-means On Window Features

Supervised learning uses labels. Unsupervised learning finds structure from feature similarity.

Here we use **k-means**, which groups rows or windows into clusters based on similarity in feature space. The cluster IDs themselves do not carry meaning ahead of time. We interpret them *after* fitting by looking at:

- how many examples ended up in each cluster,
- the mean issue rate inside each cluster,
- where the cluster sits in feature space,
- and what the underlying time-series examples look like.

This is useful when you want to surface interesting operating regimes or suspicious periods even before a label model is mature.

One subtle point: the cluster numbers and colours are arbitrary. They only become meaningful after we interpret them using the plots and summary statistics below.


### 🔎 k-means Settings

Main variables:

- `n_clusters`: how many clusters we ask the algorithm to find.
- `n_init`: how many random initializations to try. `"auto"` is a good default in current scikit-learn.
- `KMEANS_FEATURE_MODE`: inherited from the dataset profile. Some datasets cluster best on window summaries, while spike-driven ones may cluster better on row-level features.

The number of cached windows loaded for clustering is derived from `DATA_FRACTION`. If you want to experiment with k-means itself, the most useful first change is usually `n_clusters`.


In [ ]:
KMEANS_CONFIG = {
    "n_clusters": 5,
    "n_init": "auto",
}
# KMEANS_WINDOW_LIMIT is defined here because it only affects k-means.
# It caps the number of cached windows/rows passed into clustering.
# Set KMEANS_WINDOW_LIMIT = None after this block if you want to cluster every selected window/row.
KMEANS_WINDOW_BASE_ROWS = 5_000  # Rows/windows used when DATA_FRACTION = 1.0 and caps are on.
KMEANS_WINDOW_MIN_ROWS = 2_000   # Smallest k-means cap after scaling by DATA_FRACTION.
if ML_STATE['USE_DATA_FRACTION_BUDGETS'] and ML_STATE['DATA_FRACTION'] < 0.999:
    KMEANS_WINDOW_LIMIT = max(KMEANS_WINDOW_MIN_ROWS, int(KMEANS_WINDOW_BASE_ROWS * ML_STATE['DATA_FRACTION']))
else:
    KMEANS_WINDOW_LIMIT = None

# Example override:
# KMEANS_WINDOW_LIMIT = 10_000
KMEANS_EXAMPLES_PER_CLUSTER = 1
KMEANS_EXAMPLE_CONTEXT_POINTS = 7500
KMEANS_EXAMPLE_HIGHLIGHT_ALPHA = 0.22

display(pd.Series(KMEANS_CONFIG, name="value").rename_axis("kmeans_parameter").to_frame())
print(
    {
        "DATA_FRACTION": ML_STATE['DATA_FRACTION'],
        "USE_DATA_FRACTION_BUDGETS": ML_STATE['USE_DATA_FRACTION_BUDGETS'],
        "KMEANS_FEATURE_MODE": ML_STATE['KMEANS_FEATURE_MODE'],
        "KMEANS_WINDOW_LIMIT": KMEANS_WINDOW_LIMIT,
        "KMEANS_EXAMPLES_PER_CLUSTER": KMEANS_EXAMPLES_PER_CLUSTER,
        "KMEANS_EXAMPLE_CONTEXT_POINTS": KMEANS_EXAMPLE_CONTEXT_POINTS,
        "KMEANS_EXAMPLE_HIGHLIGHT_ALPHA": KMEANS_EXAMPLE_HIGHLIGHT_ALPHA,
    }
)


### Fit k-means

The settings cell above chooses the number of clusters and the row/window cap. The next cell builds the feature table, standardises the numeric features, fits k-means, and summarises how each cluster lines up with known issue labels.


In [ ]:
if ML_STATE['KMEANS_FEATURE_MODE'] == "window_summary":
    window_df = pd.read_parquet(ML_STATE['window_cache_path'], columns=ML_STATE['WINDOW_USE_COLUMNS'])
    window_df["window_start"] = pd.to_datetime(window_df["window_start"], utc=True)
    window_df["window_end"] = pd.to_datetime(window_df["window_end"], utc=True)
    window_df = (
        window_df[window_df["source_file"].isin(ML_STATE['selected_source_files'])]
        .sort_values("window_start")
        .reset_index(drop=True)
    )
    window_df = evenly_spaced_take(window_df, KMEANS_WINDOW_LIMIT, time_column="window_start")
    cluster_source_df = window_df
    kmeans_feature_columns = [
        column
        for column in cluster_source_df.columns
        if column.endswith("_mean") or column.endswith("_std")
    ]
    cluster_x_column = f"{ML_STATE['PLOT_MEASUREMENT_COLUMN']}_mean"
    cluster_y_column = f"{ML_STATE['PLOT_SECONDARY_COLUMN']}_mean"
    cluster_axis_label_suffix = "mean"
else:
    # Row-level k-means should use the same visible row budget as window-summary k-means.
    # Without this sample, row-level feature mode would materialise the full reviewed dataset.
    cluster_selection_df = evenly_spaced_take(
        ML_STATE['reviewed_model_df'][["Time UTC", "source_file"]].copy(),
        KMEANS_WINDOW_LIMIT,
        time_column="Time UTC",
    )
    cluster_source_rows = load_selected_row_level_frame(
        ML_STATE['selected_paths'],
        cluster_selection_df,
        columns=ML_STATE['ROW_USE_COLUMNS'],
    )
    cluster_source_df, feature_columns, _ = build_model_frame(
        cluster_source_rows,
        target_flag=ML_STATE['TARGET_FLAG'],
        measurement_columns=ML_STATE['measurement_columns'],
        task_mode=ML_STATE['task_mode'],
        good_labels=ML_STATE['GOOD_LABELS'],
        issue_labels=ML_STATE['ISSUE_LABELS'],
        model_row_limit=None,
    )
    cluster_source_df["issue"] = cluster_source_df["issue"].astype(int)
    cluster_source_df["window_start"] = pd.to_datetime(cluster_source_df["Time UTC"], utc=True)
    cluster_source_df["window_end"] = pd.to_datetime(cluster_source_df["Time UTC"], utc=True)
    cluster_source_df["issue_rate"] = cluster_source_df["issue"].astype(float)
    kmeans_feature_columns = [column for column in feature_columns if column in cluster_source_df.columns]
    cluster_x_column = ML_STATE['PLOT_MEASUREMENT_COLUMN']
    cluster_y_column = ML_STATE['PLOT_SECONDARY_COLUMN']
    cluster_axis_label_suffix = "value"

if cluster_x_column not in cluster_source_df.columns or cluster_y_column not in cluster_source_df.columns:
    raise ValueError(
        f"k-means plotting columns are missing for profile {ML_STATE['DATASET_PROFILE_ID']}: "
        f"{cluster_x_column!r}, {cluster_y_column!r}"
    )

window_df, cluster_summary = fit_kmeans(
    cluster_source_df,
    n_clusters=KMEANS_CONFIG["n_clusters"],
    seed=ML_STATE['SEED'],
    n_init=KMEANS_CONFIG["n_init"],
    feature_mode=ML_STATE['KMEANS_FEATURE_MODE'],
    feature_columns=kmeans_feature_columns,
)
display(cluster_summary.round({"mean_issue_rate": 3, "max_issue_rate": 3, "avg_distance": 3}))


### Read The Cluster Summary

Cluster numbers are just names until we interpret them. The next plots show where the clusters sit in feature space and whether any clusters have noticeably higher issue rates.


In [ ]:
# Plot a readable sample of clustered points so the scatter does not become an unreadable blob.
plot_window_df = evenly_spaced_take(window_df, 5000, time_column="window_start")
cluster_ids = sorted(cluster_summary.index.tolist())
cluster_colors = plt.cm.tab10(np.linspace(0, 1, len(cluster_ids)))
cluster_palette = {cluster_id: cluster_colors[idx] for idx, cluster_id in enumerate(cluster_ids)}

fig, axes = plt.subplots(1, 2, figsize=(17, 6))
cluster_centers = (
    plot_window_df.groupby("cluster")[[cluster_x_column, cluster_y_column]]
    .mean()
    .reindex(cluster_ids)
)

legend_handles = []
for cluster_id in cluster_ids:
    subset = plot_window_df[plot_window_df["cluster"] == cluster_id]
    axes[0].scatter(
        subset[cluster_x_column],
        subset[cluster_y_column],
        s=18,
        alpha=0.55,
        color=cluster_palette[cluster_id],
        edgecolors="none",
    )
    axes[0].scatter(
        cluster_centers.loc[cluster_id, cluster_x_column],
        cluster_centers.loc[cluster_id, cluster_y_column],
        s=220,
        marker="*",
        color=cluster_palette[cluster_id],
        edgecolors="black",
        linewidths=0.8,
    )
    legend_handles.append(
        Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor=cluster_palette[cluster_id],
            markersize=8,
            label=(
                f"Cluster {cluster_id} | n={int(cluster_summary.loc[cluster_id, 'window_count'])} | "
                f"mean issue={cluster_summary.loc[cluster_id, 'mean_issue_rate']:.2f}"
            ),
        )
    )

axes[0].set_title(f"k-means clusters in {ML_STATE['KMEANS_FEATURE_MODE'].replace('_', ' ')} feature space")
axes[0].set_xlabel(f"{ML_STATE['PLOT_MEASUREMENT_COLUMN']} ({cluster_axis_label_suffix})")
axes[0].set_ylabel(f"{ML_STATE['PLOT_SECONDARY_COLUMN']} ({cluster_axis_label_suffix})")
axes[0].grid(alpha=0.25)
axes[0].legend(handles=legend_handles, title="Cluster legend", loc="upper left", bbox_to_anchor=(1.02, 1.0), frameon=True)

bar_positions = np.arange(len(cluster_ids))
bar_values = [cluster_summary.loc[cluster_id, "mean_issue_rate"] for cluster_id in cluster_ids]
bar_colors = [cluster_palette[cluster_id] for cluster_id in cluster_ids]
axes[1].bar(bar_positions, bar_values, color=bar_colors, edgecolor="black", linewidth=0.6)
axes[1].set_xticks(bar_positions)
axes[1].set_xticklabels([f"Cluster {cluster_id}" for cluster_id in cluster_ids], rotation=30)
axes[1].set_ylim(0, 1.05)
axes[1].set_ylabel("Mean issue rate")
axes[1].set_title("Average issue rate by cluster")
for idx, cluster_id in enumerate(cluster_ids):
    axes[1].text(
        idx,
        bar_values[idx] + 0.02,
        f"n={int(cluster_summary.loc[cluster_id, 'window_count'])}\n{bar_values[idx]:.1%}",
        ha="center",
        va="bottom",
        fontsize=9,
    )

plt.tight_layout()
plt.show()

interesting_windows = window_df.sort_values(
    ["issue_rate", "distance_to_centroid"],
    ascending=[False, False],
).head(10)
interesting_windows = interesting_windows[
    ["window_start", "window_end", "source_file", "cluster", "issue_rate", "distance_to_centroid"]
].copy()
interesting_windows["source_file"] = interesting_windows["source_file"].map(clean_source_file_label)
display(interesting_windows.round({"issue_rate": 3, "distance_to_centroid": 3}))


### Looking At Real Time-Series Windows From Each Cluster

The scatter plot shows where clusters sit in feature space, but it does not show what the underlying sensor behaviour actually looked like.

The next plot closes that loop. For each cluster, we pick a representative window that sits close to its centroid, then:

- show a wider time-series context from the original row-level parquet,
- show the true target-label regions from that underlying data,
- highlight the exact window used in k-means,
- mark the datapoints inside that highlighted window.

This makes it much easier to interpret clusters as real operating regimes rather than abstract colored dots.


In [ ]:
cluster_example_figure, cluster_example_windows = plot_cluster_window_examples(
    window_df,
    source_to_row_part=ML_STATE['source_to_row_part'],
    measurement_column=ML_STATE['PLOT_MEASUREMENT_COLUMN'],
    secondary_column=ML_STATE['PLOT_SECONDARY_COLUMN'],
    target_flag=ML_STATE['TARGET_FLAG'],
    good_labels=ML_STATE['GOOD_LABELS'],
    examples_per_cluster=KMEANS_EXAMPLES_PER_CLUSTER,
    context_points=KMEANS_EXAMPLE_CONTEXT_POINTS,
    highlight_alpha=KMEANS_EXAMPLE_HIGHLIGHT_ALPHA,
    flag_palette=DEFAULT_FLAG_PALETTE,
    target_display_name=ML_STATE['FLAG_EXAMPLE_DISPLAY_NAME'],
    label_meanings=ML_STATE['FLAG_EXAMPLE_LABEL_MEANINGS'],
)
plt.show()
display(cluster_example_windows)


### Date-Range Demo: See Which Clusters Appear in a Specific Interval

k-means does not predict target labels directly. Instead, it groups short windows with similar summary behaviour.

This demo lets us ask: **what kinds of windows show up inside one selected time range, and how do their issue rates differ?**


In [ ]:
KMEANS_RANGE_START = "2026-02-9T12:56:27.707000+00:00"
KMEANS_RANGE_END = "2026-02-11T12:56:27.707000+00:00"
KMEANS_AUTO_SELECT_TEST_RANGE = True
KMEANS_MAX_POINTS_TO_PLOT = 800

print(
    {
        "KMEANS_RANGE_START": KMEANS_RANGE_START,
        "KMEANS_RANGE_END": KMEANS_RANGE_END,
        "KMEANS_AUTO_SELECT_TEST_RANGE": KMEANS_AUTO_SELECT_TEST_RANGE,
        "KMEANS_MAX_POINTS_TO_PLOT": KMEANS_MAX_POINTS_TO_PLOT,
    }
)


The previous cell chooses a UTC range for the k-means inspection. The next cell reloads that interval, finds the matching clustered windows, and overlays cluster bands on the time series so we can inspect behaviour in context.


In [ ]:
kmeans_range_selection = select_time_range(
    ML_STATE['test_df'],
    time_column="Time UTC",
    label_column=ML_STATE['TARGET_FLAG'],
    start=KMEANS_RANGE_START,
    end=KMEANS_RANGE_END,
    auto_select=KMEANS_AUTO_SELECT_TEST_RANGE,
    max_points=KMEANS_MAX_POINTS_TO_PLOT,
)

kmeans_interval_rows = load_rows_for_time_range(
    ML_STATE['metadata'],
    row_cache_dir=Path(ML_STATE["ROW_CACHE_DIR"]),
    start=kmeans_range_selection["start"],
    end=kmeans_range_selection["end"],
    columns=ML_STATE['ROW_USE_COLUMNS'],
)
kmeans_interval_windows = window_df[
    (window_df["window_start"] <= kmeans_range_selection["end"])
    & (window_df["window_end"] >= kmeans_range_selection["start"])
].copy()

if kmeans_interval_rows.empty or kmeans_interval_windows.empty:
    print("No overlapping k-means windows were found in the requested range.")
else:
    kmeans_interval_origin = infer_interval_origin(
        kmeans_range_selection["start"],
        kmeans_range_selection["end"],
        {"train": ML_STATE['train_df'], "validation": ML_STATE['valid_df'], "test": ML_STATE['test_df']},
    )
    kmeans_interval_intervals = merge_adjacent_intervals(
        kmeans_interval_windows.rename(
            columns={"window_start": "start", "window_end": "end", "cluster": "label"}
        )[["start", "end", "label"]]
    )
    kmeans_cluster_counts = (
        kmeans_interval_windows.groupby("cluster")
        .agg(
            window_count=("cluster", "size"),
            mean_issue_rate=("issue_rate", "mean"),
            max_issue_rate=("issue_rate", "max"),
        )
        .sort_index()
    )

    print(
        {
            "selection_mode": kmeans_range_selection["selection_mode"],
            "selected_priority_flag": kmeans_range_selection["selected_label"],
            "interval_origin": kmeans_interval_origin,
            "range_start": kmeans_range_selection["start"].isoformat(),
            "range_end": kmeans_range_selection["end"].isoformat(),
            "row_points_in_interval": int(len(kmeans_interval_rows)),
            "window_count_in_interval": int(len(kmeans_interval_windows)),
        }
    )
    display(kmeans_cluster_counts.round(3))

    kmeans_demo_figure = plot_time_series_with_bands(
        kmeans_interval_rows,
        band_specs=[
            {
                "title": "k-means clusters",
                "intervals": kmeans_interval_intervals,
                "palette": cluster_palette,
            }
        ],
        measurement_column=ML_STATE['PLOT_MEASUREMENT_COLUMN'],
        secondary_column=ML_STATE['PLOT_SECONDARY_COLUMN'],
        max_points=KMEANS_MAX_POINTS_TO_PLOT,
        title="k-means cluster assignments on a selected time range",
        legend_title="cluster",
    )
    plt.show()


### Reflection Questions: k-means

1. Do these cluster spans look like real operating regimes, or do some clusters still seem hard to interpret physically?
2. If you changed `n_clusters`, which regions in this interval do you expect would split apart or merge together?
3. How closely do the cluster spans line up with target-label changes, and what does that say about the usefulness of unsupervised learning here?

Add your own notes or follow-up questions here:

- 
- 
- 


## Part 3 — 1D CNN


### Sequential Model: 1D CNN

A **convolutional neural network (CNN)** applies small learnable filters across an ordered signal. For images that order is two-dimensional. In this notebook, the order is **time**.

The key idea is:

1. a short filter looks at a local pattern,
2. the same filter slides across the whole sequence,
3. later layers combine those detected patterns into a final prediction.

For this scalar QC task, a 1D CNN can learn patterns such as:

- sudden spikes or drops,
- flat segments,
- repeated local shapes across several sensor channels.

Why this is useful here:

- Random Forest treats each row mostly as a tabular snapshot,
- CNN keeps a short stretch of signal together as one example,
- the training loop also gives us a chance to teach batching, validation checkpoints, and early stopping.

The CNN section runs by default in this notebook and follows standard training practice:

- train / validation / test split,
- class-aware loss weighting,
- best-checkpoint saving based on validation F1,
- early stopping,
- mini-batch training through `DataLoader`,
- pinned-memory and multi-worker loading when the local machine supports it.

Useful references: [PyTorch tutorial on defining neural networks](https://docs.pytorch.org/tutorials/beginner/blitz/neural_networks_tutorial), [PyTorch DataLoader docs](https://pytorch.org/docs/stable/data.html), [PyTorch performance tuning guide](https://pytorch.org/tutorials/recipes/recipes/tuning_guide.html)


![Convolutional network diagram](../assets/session1/convolutional_network.png)

Diagram idea: small filters slide across the input, produce feature maps, and later layers combine those maps into a final prediction.

In this notebook the same logic is applied to **1D time windows** rather than 2D images.

Image credit: Aphex34, Wikimedia Commons, CC BY-SA 4.0. Source: [File:Convolutional Network.png](https://commons.wikimedia.org/wiki/File:Convolutional_Network.png)


### 🎛️ CNN Settings

These settings control the baseline 1D CNN and its training loop.

Model-shape variables:

- `window_size`: number of time steps in each training window.
- `output_mode`: choose `"window"` for one label per window or `"per_timestep"` for one label at every time step.
- `conv_channels`: number of filters in each convolution layer.
- `dropout`: dropout probability used for regularisation.
- `label_reduction`: how row-level labels become one window label. This only matters when `output_mode="window"`.

Optimisation variables:

- `epochs`: maximum number of passes through the training data.
- `batch_size`: number of windows per optimisation step.
- `learning_rate`: optimiser step size.
- `weight_decay`: L2-style regularisation for the optimiser.
- `patience` and `min_delta`: early-stopping controls.
- `lr_decay_factor` and `lr_patience`: learning-rate scheduler controls.
- `gradient_clip_norm`: cap on gradient size to stabilize training.

DataLoader variables:

- `num_workers`: worker processes for loading batches.
- `pin_memory`: can speed up host-to-GPU transfer.
- `persistent_workers`: keeps workers alive between epochs.
- `prefetch_factor`: how many batches each worker prepares ahead of time.

Dataset-aware default:

- the turbidity and conductivity-plug profiles start in `"per_timestep"` mode because short local events are easy to lose inside mixed windows,
- the other profiles start in `"window"` mode to keep the first baseline simpler.


In [ ]:
# CNN_CONFIG controls the learning problem, model size, and training loop.
# Start with these defaults, then change one or two values at a time so results stay interpretable.
CNN_CONFIG = {
    # Number of consecutive timestamps given to the CNN for one example.
    "window_size": 128,
    # "window" predicts one label for the whole window; "per_timestep" predicts one label per row.
    "output_mode": ML_STATE['DEFAULT_SEQUENCE_OUTPUT_MODE'],
    "epochs": 10,
    "batch_size": 128,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "patience": 3,
    "min_delta": 0.001,
    "dropout": 0.2,
    "lr_decay_factor": 0.5,
    "lr_patience": 1,
    "gradient_clip_norm": 1.0,
    # Each number is the number of filters in a Conv1D layer. More filters can learn richer patterns but cost more memory.
    "conv_channels": [32, 64],
    "label_reduction": "worst",
}

# DataLoader settings control how batches are moved into PyTorch.
# num_workers=0 is slower but safest inside shared notebook environments.
_loader_torch = globals().get("torch", None)
CNN_LOADER_CONFIG = {
    "num_workers": 0,
    "pin_memory": bool(_loader_torch is not None and _loader_torch.cuda.is_available()),
    "persistent_workers": False,
    "prefetch_factor": 2,
}

display(pd.Series(CNN_CONFIG, name="value").rename_axis("cnn_parameter").to_frame())
display(pd.Series(CNN_LOADER_CONFIG, name="value").rename_axis("loader_parameter").to_frame())


### Utility For Window Labels

The CNN and transformer sections now support two output styles:

- `"window"`: one prediction for the whole window,
- `"per_timestep"`: one prediction for every point inside the window.

When you choose `"window"`, we need one short rule for turning several row-level target labels inside a window into one label. The helper below keeps that reduction in one place so the later cells stay easier to read.


In [ ]:
# Reduce row-level labels to one window-level label for sequence models.
def reduce_window_target(
    values: np.ndarray,
    mode: str,
    severity_order: tuple[int, ...] = (1, 3, 4, 9),
) -> int:
    labels = [int(value) for value in values if pd.notna(value)]
    if not labels:
        return severity_order[0]
    effective_order = tuple(sorted(set(severity_order).union(labels)))
    severity_rank = {label: index for index, label in enumerate(effective_order)}
    if mode == "worst":
        return max(labels, key=lambda label: severity_rank.get(label, -1))
    counts = pd.Series(labels).value_counts()
    tied_labels = counts[counts == counts.max()].index.tolist()
    return max(tied_labels, key=lambda label: severity_rank.get(int(label), -1))


### CNN Step 1: Turn Rows into Windows

A CNN expects a fixed-size tensor, not an arbitrary-length table.

In this step we:

- collect the measurement columns we want to model,
- group consecutive rows into fixed windows,
- either reduce many row-level target labels into one window label or keep one target per timestamp,
- split the windows with the active train / validation / test strategy,
- normalise each sensor channel using only the training split.

One important detail: the baseline CNN uses windows of the selected measurement columns as input. That gives it more temporal context than the RF, with fewer hand-engineered tabular features.

This is a good place to pause and ask: what information do we lose when we compress several row labels into one window label, and when is that simplification actually helpful?


In [ ]:
# Run this cell when you are ready to work through the CNN section.
# Earlier notebook parts do not need PyTorch, so we import it here.
try:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset

    PYTORCH_READY = True
    CNN_READY = True
    print({
        "torch": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    })
except ImportError as exc:
    PYTORCH_READY = False
    CNN_READY = False
    print("PyTorch is not installed in this environment.")
    raise SystemExit(exc)


In [ ]:
if not CNN_READY:
    CNN_RUN = False
    print("CNN training cell skipped.")
else:
    # Load the row-level measurements for the already chosen train/validation/test split.
    # This keeps the CNN evaluation aligned with the Random Forest fixed split.
    sequence_split_frames_materialized = materialize_reviewed_split_frames(
        ML_STATE['selected_paths'],
        {"train": ML_STATE['train_df'], "validation": ML_STATE['valid_df'], "test": ML_STATE['test_df']},
        columns=ML_STATE['ROW_USE_COLUMNS'],
        target_flag=ML_STATE['TARGET_FLAG'],
        task_mode=ML_STATE['task_mode'],
        good_labels=ML_STATE['GOOD_LABELS'],
        issue_labels=ML_STATE['ISSUE_LABELS'],
    )
    # Pull the settings into short variable names because they are used throughout the CNN cells.
    WINDOW_SIZE = CNN_CONFIG["window_size"]
    CNN_OUTPUT_MODE = CNN_CONFIG["output_mode"]
    # Convert rows into fixed-length windows.
    # The helper returns X arrays shaped like (windows, time, channels) and y labels for each split.
    cnn_bundle = build_sequence_split_bundle_from_frames(
        sequence_split_frames_materialized,
        measurement_columns=ML_STATE['measurement_columns'],
        target_column="model_target",
        task_mode=ML_STATE['task_mode'],
        output_mode=CNN_OUTPUT_MODE,
        window_size=WINDOW_SIZE,
        label_reduction=CNN_CONFIG["label_reduction"],
    )
    # class_labels maps model output indexes back to the original QC labels.
    class_labels = cnn_bundle.class_labels

    # A classifier needs at least two classes and at least one complete window in every split.
    if ML_STATE['task_mode'] == "multiclass" and len(class_labels) < 2:
        CNN_RUN = False
        print(
            {
                "output_mode": CNN_OUTPUT_MODE,
                "windows_total": int(len(cnn_bundle.X_train) + len(cnn_bundle.X_valid) + len(cnn_bundle.X_test)),
                "active_labels": class_labels,
                "skip_reason": "Need at least two active classes to train the CNN in multiclass mode.",
            }
        )
    elif len(cnn_bundle.X_train) == 0 or len(cnn_bundle.X_valid) == 0 or len(cnn_bundle.X_test) == 0:
        CNN_RUN = False
        print(
            {
                "output_mode": CNN_OUTPUT_MODE,
                "split_strategy": ML_STATE['FIXED_SPLIT_STRATEGY'],
                "train_subset_strategy": ML_STATE['TRAIN_SUBSET_STRATEGY'],
                "skip_reason": "At least one split produced no full windows for this configuration.",
            }
        )
    else:
        # Move to channel-first layout expected by Conv1D.
        X_train = np.transpose(cnn_bundle.X_train, (0, 2, 1))
        X_valid = np.transpose(cnn_bundle.X_valid, (0, 2, 1))
        X_test = np.transpose(cnn_bundle.X_test, (0, 2, 1))
        # Keep the labels beside the transposed feature arrays; labels are not normalised.
        y_train = cnn_bundle.y_train
        y_valid = cnn_bundle.y_valid
        y_test = cnn_bundle.y_test

        # Normalise each sensor channel using only the training split.
        # Validation and test use the training mean/std so evaluation does not peek at held-out data.
        channel_mean = X_train.mean(axis=(0, 2), keepdims=True)
        channel_std = X_train.std(axis=(0, 2), keepdims=True) + 1e-6
        X_train = (X_train - channel_mean) / channel_std
        X_valid = (X_valid - channel_mean) / channel_std
        X_test = (X_test - channel_mean) / channel_std

        CNN_RUN = True
        print(
            {
                "output_mode": CNN_OUTPUT_MODE,
                "split_strategy": ML_STATE['FIXED_SPLIT_STRATEGY'],
                "train_subset_strategy": ML_STATE['TRAIN_SUBSET_STRATEGY'],
                "fixed_split_block_rows": ML_STATE['FIXED_SPLIT_BLOCK_ROWS'] if ML_STATE['FIXED_SPLIT_STRATEGY'] in {"interleaved_blocks", "episode_aware"} else None,
                "windows_total": int(len(X_train) + len(X_valid) + len(X_test)),
                "train_windows": int(len(X_train)),
                "validation_windows": int(len(X_valid)),
                "test_windows": int(len(X_test)),
                "window_size": int(WINDOW_SIZE),
                "channels": int(len(ML_STATE['measurement_columns'])),
            }
        )


### CNN Step 2: Build the Model and the DataLoaders

Here we create four pieces:

- the neural network itself,
- the loss function,
- the optimiser and learning-rate scheduler,
- the `DataLoader` objects that stream mini-batches during training.

The output toggle changes the final prediction head:

- in `"window"` mode, the CNN pools across time and emits one label for the whole window,
- in `"per_timestep"` mode, the CNN keeps the time axis and emits one label per point.

Notice that the `DataLoader` keeps the full dataset in CPU memory and feeds the GPU one batch at a time. That is usually much more efficient than trying to move every window onto the GPU all at once.


In [ ]:
if not globals().get("CNN_RUN", False):
    print("CNN model setup skipped.")
else:
    # Fix PyTorch randomness so reruns are easier to compare.
    torch.manual_seed(ML_STATE['SEED'])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cuda":
        torch.backends.cudnn.benchmark = True

    # TinyQCNet is a compact 1D CNN: it scans across time and learns short temporal patterns.
    # Input tensors have shape (batch, sensor_channels, time_steps).
    class TinyQCNet(nn.Module):
        def __init__(self, channels: int, output_dim: int) -> None:
            super().__init__()
            self.output_dim = output_dim
            # The feature extractor applies convolution filters over neighbouring timestamps.
            self.features = nn.Sequential(
                nn.Conv1d(channels, CNN_CONFIG["conv_channels"][0], kernel_size=7, padding=3),
                nn.ReLU(),
                nn.Conv1d(CNN_CONFIG["conv_channels"][0], CNN_CONFIG["conv_channels"][1], kernel_size=5, padding=2),
                nn.ReLU(),
                nn.Dropout(CNN_CONFIG["dropout"]),
            )
            if CNN_OUTPUT_MODE == "window":
                # Pool across time when the model should emit one label for the whole window.
                self.pool = nn.AdaptiveAvgPool1d(1)
                self.head = nn.Linear(CNN_CONFIG["conv_channels"][1], output_dim)
            else:
                # A 1x1 convolution keeps the time axis and emits one prediction per timestamp.
                self.head = nn.Conv1d(CNN_CONFIG["conv_channels"][1], output_dim, kernel_size=1)

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            # First transform the raw sensor channels into learned temporal features.
            x = self.features(x)
            if CNN_OUTPUT_MODE == "window":
                x = self.pool(x).squeeze(-1)
                return self.head(x)
            logits = self.head(x)
            if self.output_dim == 1:
                return logits.squeeze(1)
            return logits.transpose(1, 2)

    # Choose a loss function that matches the target type.
    # Multiclass uses one integer class per example; binary uses a single logit and class weighting.
    if ML_STATE['task_mode'] == "multiclass":
        train_targets_tensor = torch.from_numpy(y_train).long()
        valid_targets_tensor = torch.from_numpy(y_valid).long()
        test_targets_tensor = torch.from_numpy(y_test).long()
        # Weight rare classes more heavily so the model does not learn to ignore issue labels.
        class_counts = np.bincount(y_train.reshape(-1), minlength=len(class_labels)).clip(min=1)
        class_weights = y_train.size / (len(class_counts) * class_counts)
        loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=device))
        output_dim = len(class_labels)
    else:
        train_targets_tensor = torch.from_numpy(y_train.astype(np.float32))
        valid_targets_tensor = torch.from_numpy(y_valid.astype(np.float32))
        test_targets_tensor = torch.from_numpy(y_test.astype(np.float32))
        # pos_weight gives the issue class more influence when issues are rare.
        positive_count = max(float(y_train.sum()), 1.0)
        negative_count = max(float(y_train.size - y_train.sum()), 1.0)
        pos_weight = torch.tensor([negative_count / positive_count], dtype=torch.float32, device=device)
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        output_dim = 1

    # TensorDataset pairs each input window with its target label for DataLoader batching.
    train_dataset = TensorDataset(torch.from_numpy(np.ascontiguousarray(X_train)), train_targets_tensor)
    valid_dataset = TensorDataset(torch.from_numpy(np.ascontiguousarray(X_valid)), valid_targets_tensor)
    test_dataset = TensorDataset(torch.from_numpy(np.ascontiguousarray(X_test)), test_targets_tensor)

    # DataLoader options are collected once so train/validation/test loaders stay consistent.
    loader_kwargs = {}
    num_workers = int(CNN_LOADER_CONFIG["num_workers"])
    if num_workers > 0:
        loader_kwargs["num_workers"] = num_workers
        loader_kwargs["persistent_workers"] = bool(CNN_LOADER_CONFIG["persistent_workers"])
        loader_kwargs["prefetch_factor"] = int(CNN_LOADER_CONFIG["prefetch_factor"])
    if CNN_LOADER_CONFIG["pin_memory"] and device.type == "cuda":
        loader_kwargs["pin_memory"] = True
    use_non_blocking = bool(loader_kwargs.get("pin_memory", False) and device.type == "cuda")

    # The notebook no longer needs the large NumPy arrays after TensorDatasets are built.
    # Deleting them reduces memory pressure before training starts.
    del X_train, X_valid, X_test, y_train, y_valid, y_test
    del train_targets_tensor, valid_targets_tensor, test_targets_tensor
    import gc
    gc.collect()

    train_loader = DataLoader(
        train_dataset,
        batch_size=CNN_CONFIG["batch_size"],
        shuffle=True,
        **loader_kwargs,
    )
    valid_loader = DataLoader(
        valid_dataset,
        batch_size=CNN_CONFIG["batch_size"],
        shuffle=False,
        **loader_kwargs,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=CNN_CONFIG["batch_size"],
        shuffle=False,
        **loader_kwargs,
    )

    # Create the model and optimiser. AdamW is a good default for neural networks with weight decay.
    model = TinyQCNet(channels=len(ML_STATE['measurement_columns']), output_dim=output_dim).to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=CNN_CONFIG["learning_rate"],
        weight_decay=CNN_CONFIG["weight_decay"],
    )
    # Reduce the learning rate when validation F1 stops improving.
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=CNN_CONFIG["lr_decay_factor"],
        patience=CNN_CONFIG["lr_patience"],
    )

    # These variables track the best validation checkpoint and support early stopping.
    best_metric = -np.inf
    best_epoch = 0
    patience_counter = 0
    history = []
    best_state = None

    # One shared epoch function handles both training and evaluation.
    # training=True enables gradients and optimiser steps; training=False only scores batches.
    def run_epoch(loader, training: bool) -> tuple[float, np.ndarray, np.ndarray]:
        model.train(training)
        total_loss = 0.0
        predictions = []
        targets = []
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device, non_blocking=use_non_blocking)
            batch_y = batch_y.to(device, non_blocking=use_non_blocking)
            with torch.set_grad_enabled(training):
                logits = model(batch_x)
                # The logits shape differs for window-level vs per-timestep predictions,
                # so multiclass loss receives the class dimension in the place PyTorch expects.
                if ML_STATE['task_mode'] == "multiclass":
                    if CNN_OUTPUT_MODE == "window":
                        loss = loss_fn(logits, batch_y)
                        batch_predictions = logits.argmax(dim=1)
                    else:
                        loss = loss_fn(logits.permute(0, 2, 1), batch_y)
                        batch_predictions = logits.argmax(dim=-1)
                else:
                    loss = loss_fn(logits, batch_y)
                    batch_predictions = (torch.sigmoid(logits) >= 0.5).long()
                if training:
                    # Clear old gradients, backpropagate this batch, optionally clip, then update weights.
                    optimizer.zero_grad(set_to_none=True)
                    loss.backward()
                    if CNN_CONFIG["gradient_clip_norm"]:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), CNN_CONFIG["gradient_clip_norm"])
                    optimizer.step()
            total_loss += float(loss.item()) * len(batch_x)
            # Store flattened predictions/targets so F1 can be computed across the whole split.
            predictions.append(batch_predictions.detach().cpu().reshape(-1).numpy())
            targets.append(batch_y.detach().cpu().reshape(-1).numpy())
        predictions_array = np.concatenate(predictions) if predictions else np.array([])
        targets_array = np.concatenate(targets) if targets else np.array([])
        average_loss = total_loss / max(len(loader.dataset), 1)
        return average_loss, predictions_array, targets_array

    print(
        {
            "device": str(device),
            "output_mode": CNN_OUTPUT_MODE,
            "train_batches": len(train_loader),
            "validation_batches": len(valid_loader),
            "test_batches": len(test_loader),
            "loader_config": CNN_LOADER_CONFIG,
        }
    )


### CNN Step 3: Train with Validation Monitoring

This cell runs the optimisation loop.

We are following standard training practice:

- train on the training split,
- score the model on the validation split after each epoch,
- save the best checkpoint seen so far,
- stop early if the validation metric stops improving.

Watch the printed validation F1 as the run progresses. When `output_mode="window"`, that F1 is computed per window. When `output_mode="per_timestep"`, it is computed across all timestamps in the validation windows.


In [ ]:
if not globals().get("CNN_RUN", False):
    print("CNN fit skipped.")
else:
    # Train for a small number of epochs and evaluate validation performance after each pass.
    for epoch in range(1, CNN_CONFIG["epochs"] + 1):
        # The training pass updates model weights; predictions are not needed here.
        train_loss, _, _ = run_epoch(train_loader, training=True)
        # The validation pass does not update weights; it estimates generalisation during training.
        valid_loss, valid_preds, valid_targets = run_epoch(valid_loader, training=False)
        valid_f1 = f1_score(valid_targets, valid_preds, average=report_average(ML_STATE['task_mode']), zero_division=0)
        # Validation F1 drives the learning-rate scheduler and the best-checkpoint decision.
        scheduler.step(valid_f1)
        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "valid_loss": valid_loss,
                "valid_f1": valid_f1,
                "learning_rate": optimizer.param_groups[0]["lr"],
            }
        )
        print(
            f"Epoch {epoch:>2} | train_loss={train_loss:.4f} | valid_loss={valid_loss:.4f} | valid_f1={valid_f1:.4f}"
        )

        # Save a checkpoint whenever validation F1 improves by at least min_delta.
        if valid_f1 > best_metric + CNN_CONFIG["min_delta"]:
            best_metric = valid_f1
            best_epoch = epoch
            patience_counter = 0
            best_state = copy.deepcopy(model.state_dict())
            torch.save(
                {
                    "model_state_dict": best_state,
                    "task_mode": ML_STATE['task_mode'],
                    "class_labels": class_labels,
                    "feature_columns": ML_STATE['measurement_columns'],
                    "window_size": WINDOW_SIZE,
                    "output_mode": CNN_OUTPUT_MODE,
                    "config": {**CNN_CONFIG, **CNN_LOADER_CONFIG},
                },
                ML_STATE["CNN_MODEL_PATH"],
            )
        else:
            # Early stopping protects the notebook from spending epochs on a stalled model.
            patience_counter += 1
            if patience_counter >= CNN_CONFIG["patience"]:
                print(f"Early stopping triggered at epoch {epoch}")
                break


### CNN Step 4: Reload the Best Checkpoint and Evaluate

Training loss alone is not enough. In this final step we:

- reload the weights that gave the best validation F1,
- run the held-out test split once,
- inspect the classification report and confusion matrix.

This keeps the evaluation honest and makes it clear that the validation set guided model selection, while the test set is reserved for final reporting.


In [ ]:
if not globals().get("CNN_RUN", False):
    print("CNN evaluation skipped.")
else:
    # Reload the best validation checkpoint before touching the test split.
    if best_state is None:
        raise RuntimeError("CNN training did not produce a valid checkpoint")

    model.load_state_dict(best_state)
    # Test results are reported once, after model selection is finished.
    test_loss, test_preds, test_targets = run_epoch(test_loader, training=False)

    print(pd.DataFrame(history).tail())
    print(
        {
            "output_mode": CNN_CONFIG["output_mode"],
            "windows": len(train_dataset) + len(valid_dataset) + len(test_dataset),
            "device": str(device),
            "best_validation_f1": float(best_metric),
            "best_epoch": best_epoch,
            "test_loss": float(test_loss),
            "saved_model": str(ML_STATE["CNN_MODEL_PATH"]),
            "loader_config": CNN_LOADER_CONFIG,
        }
    )

    # Build human-readable report labels for either multiclass or binary mode.
    if ML_STATE['task_mode'] == "multiclass":
        report_labels = list(range(len(class_labels)))
        report_names = [str(label) for label in class_labels]
    else:
        report_labels = [0, 1]
        report_names = ["0", "1"]

    print(
        classification_report(
            test_targets,
            test_preds,
            labels=report_labels,
            target_names=report_names,
            zero_division=0,
        )
    )

    # Plot learning curves and a normalised confusion matrix to show both training behaviour and errors.
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    history_frame = pd.DataFrame(history)
    axes[0].plot(history_frame["epoch"], history_frame["train_loss"], label="train")
    axes[0].plot(history_frame["epoch"], history_frame["valid_loss"], label="validation")
    axes[0].set_title("CNN loss history")
    axes[0].legend()
    ConfusionMatrixDisplay.from_predictions(
        test_targets,
        test_preds,
        labels=report_labels,
        display_labels=report_names,
        normalize="true",
        xticks_rotation=45,
        ax=axes[1],
    )
    axes[1].set_title("CNN test confusion matrix")
    plt.tight_layout()
    plt.show()


### Date-Range Demo: See CNN Predictions on a Specific Interval

This demo follows the same `output_mode` toggle as the training section:

- `"window"` shows one true/predicted label span per window,
- `"per_timestep"` shows one true/predicted label span per timestamp.


In [ ]:
# Leave start/end as None to auto-select an interesting test interval.
# Set explicit timestamps here when you want to inspect a specific deployment period.
CNN_RANGE_START = None
CNN_RANGE_END = None
CNN_AUTO_SELECT_TEST_RANGE = True
CNN_MAX_POINTS_TO_PLOT = 1024

print(
    {
        "CNN_RANGE_START": CNN_RANGE_START,
        "CNN_RANGE_END": CNN_RANGE_END,
        "CNN_AUTO_SELECT_TEST_RANGE": CNN_AUTO_SELECT_TEST_RANGE,
        "CNN_MAX_POINTS_TO_PLOT": CNN_MAX_POINTS_TO_PLOT,
    }
)


In [ ]:
cnn_interval_metrics = None

# Keep this guard because `model`, `device`, `channel_mean`, and `channel_std`
# are created only when the CNN training cells have run successfully.
if not globals().get("CNN_RUN", False):
    cnn_demo_result = show_cnn_interval_demo(cnn_run=False)
else:
    cnn_demo_result = show_cnn_interval_demo(
        cnn_run=globals().get("CNN_RUN", False),
        # CNN_CONFIG controls the prediction shape and runtime:
        # - output_mode="window" gives one label per window;
        #   output_mode="per_timestep" gives one label per timestamp.
        # - window_size changes how much history each CNN example sees.
        # - batch_size changes how many examples are predicted at once.
        cnn_config=CNN_CONFIG,
        # These objects come from the CNN training cells above.
        model=model,
        device=device,
        channel_mean=channel_mean,
        channel_std=channel_std,
        # The helper auto-selects from `test_df` by default so this visual check
        # usually stays on held-out data. Set CNN_RANGE_START/CNN_RANGE_END in
        # the previous cell when you want to inspect a specific time period.
        train_df=ML_STATE['train_df'],
        valid_df=ML_STATE['valid_df'],
        test_df=ML_STATE['test_df'],
        range_start=CNN_RANGE_START,
        range_end=CNN_RANGE_END,
        auto_select_test_range=CNN_AUTO_SELECT_TEST_RANGE,
        max_points_to_plot=CNN_MAX_POINTS_TO_PLOT,
        # These inputs tell the helper how to reload the selected raw rows and
        # rebuild the same model-ready columns used during training.
        metadata=ML_STATE['metadata'],
        row_cache_dir=ML_STATE["ROW_CACHE_DIR"],
        row_use_columns=ML_STATE['ROW_USE_COLUMNS'],
        target_flag=ML_STATE['TARGET_FLAG'],
        measurement_columns=ML_STATE['measurement_columns'],
        task_mode=ML_STATE['task_mode'],
        class_labels=class_labels,
        # Plot labels should match the active dataset. For conductivity-plug
        # runs this means custom `ml_label` meanings rather than generic QC flags.
        plot_measurement_column=ML_STATE['PLOT_MEASUREMENT_COLUMN'],
        plot_secondary_column=ML_STATE['PLOT_SECONDARY_COLUMN'],
        flag_display_name=ML_STATE['FLAG_EXAMPLE_DISPLAY_NAME'],
        label_meanings=ML_STATE['FLAG_EXAMPLE_LABEL_MEANINGS'],
        flag_palette=DEFAULT_FLAG_PALETTE,
    )
    cnn_interval_metrics = cnn_demo_result["interval_metrics"]


### Reflection Questions: Baseline CNN

1. Which output mode is active here: one label per window or one label per timestamp? How does that change what each mistake means?
2. If `output_mode="window"`, does compressing a full window into one label hide short events? If `output_mode="per_timestep"`, are the predicted boundaries sharp enough to be useful?
3. Would you improve this result first by changing `window_size`, the window label rule, the input features, or the CNN architecture/training settings?

Add your own notes or follow-up questions here:

- 
- 
- 


## Part 4 — Transformer


### Sequential Model: Small Transformer Encoder

If you have used ChatGPT or seen a text-to-image model, you have already interacted with a **transformer**.

Transformers were introduced in 2017 in *Attention Is All You Need*, and they quickly became the dominant architecture for sequence modelling. They are most famous in language, but the underlying math is not tied to text. The same ideas are useful for scalar sensor streams, audio, and video.

A simple reason they matter is that they replaced the older "read one step, then the next step" style used by RNNs and LSTMs.

- **RNN/LSTM:** process sequences step by step, which makes training slower and can make long-range context fade.
- **Transformer:** processes the full window together and learns which positions should pay attention to which other positions.

That core operation is called **self-attention**. In plain language, each timestep asks: *which other timesteps in this window are most relevant to understanding me right now?*

For ONC-style data, that is appealing whenever a local measurement only makes sense in a wider context, such as linking a short anomaly to something that happened much earlier or later in the same window.

In this notebook we use a small **encoder-only transformer** for classification. We are not generating text, so we only keep the encoder side and attach a classifier on top.

If you want a fuller explanation after this section, here are the most useful references:

- [ONC Confluence: Transformers overview](https://internal.oceannetworks.ca/x/mwDcDg)
- [3blue1brown: Attention in transformers, step-by-step](https://www.youtube.com/watch?v=eMlx5fFNoYc)
- [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/)
- [Attention Is All You Need](https://arxiv.org/abs/1706.03762)


![Transformer model architecture](../assets/session1/transformer_model_architecture.png)

This is the big-picture map. The original transformer has:

- an **encoder stack** on the left,
- a **decoder stack** on the right.

In this notebook we only use the **encoder stack** and attach a classifier on top. So when you look at the rest of the section, focus mostly on the left half of this picture rather than trying to memorize the whole diagram.

Image credit: Arz, Wikimedia Commons, CC BY-SA 4.0. Source: [File:The-Transformer-model-architecture.png](https://commons.wikimedia.org/wiki/File:The-Transformer-model-architecture.png)


![Detailed encoder self-attention diagram](../assets/session1/transformer_encoder_self_attention_detailed.png)

This picture zooms in on the core idea of **self-attention**.

You do not need to memorize every symbol here. The important story is:

- each position creates a few learned views of itself,
- those views are compared against other positions,
- the model turns those comparisons into attention weights,
- then it blends information from the whole window into a new representation.

In plain language: a timestep can decide which other timesteps are worth listening to.

If you want the clearest visual explanation of this idea, the 3blue1brown video linked above is the best companion resource for this section.

Image credit: dvgodoy, Wikimedia Commons, CC BY-SA 4.0. Source: [File:Encoder self-attention, detailed diagram.png](https://commons.wikimedia.org/wiki/File:Encoder_self-attention,_detailed_diagram.png)


![Positional encoding heatmap](../assets/session1/transformer_positional_encoding.png)

One natural question is: if attention can compare all positions to all other positions, how does the model know **where** each timestep sits in the sequence?

That is the job of **positional encoding**.

This heatmap shows that each position gets its own pattern added to the input representation. That gives the transformer access to order information, so it can tell the difference between "this happened early in the window" and "this happened later".

Image credit: Mdbartosz, Wikimedia Commons, CC BY-SA 4.0. Source: [File:Positional encoding.png](https://commons.wikimedia.org/wiki/File:Positional_encoding.png)


### Transformer Settings

Main model variables:

- `window_size`: number of time steps in each window.
- `output_mode`: choose `"window"` for one label per window or `"per_timestep"` for one label at every time step.
- `d_model`: internal embedding size for each position.
- `nhead`: number of attention heads. This must divide `d_model`.
- `num_layers`: number of stacked encoder layers.
- `dim_feedforward`: hidden size of the feed-forward sublayer inside each encoder block.
- `dropout`: dropout used inside the model.
- `pooling`: how we reduce the sequence to one vector for classification. This only matters when `output_mode="window"`.
- `label_reduction`: how row-level labels become one window label. This only matters when `output_mode="window"`.

Training variables:

- `epochs`, `batch_size`, `learning_rate`, `weight_decay`
- `patience`, `min_delta`
- `lr_decay_factor`, `lr_patience`
- `gradient_clip_norm`

Practical note:

- larger `window_size`, `d_model`, or `num_layers` can improve context capacity,
- but they also increase memory use and training time.
- `num_workers > 0` can also increase memory use a lot on Jupyter kernels, because worker
  processes may copy parts of the parent process. Keep it at `0` unless you know the
  environment has headroom.
- `pin_memory` mainly helps when you are actually training on a CUDA device.

Dataset-aware default:

- the turbidity profile starts in `"per_timestep"` mode because the most interesting events are short and windows are often mixed,
- the other profiles start in `"window"` mode so you can begin with one simpler sequence-classification baseline.

A simple mental model for the main knobs:

- `window_size` controls how much time context the model can see,
- `d_model` controls how much representational space each timestep gets,
- `nhead` controls how many different attention patterns the model can learn in parallel,
- `num_layers` controls how many times the sequence is reprocessed with attention.


In [ ]:
TRANSFORMER_CONFIG = {
    "window_size": 128,
    "output_mode": ML_STATE['DEFAULT_SEQUENCE_OUTPUT_MODE'],
    "d_model": 64,
    "nhead": 4,
    "num_layers": 2,
    "dim_feedforward": 128,
    "dropout": 0.2,
    "pooling": "mean",
    "label_reduction": "worst",
    "epochs": 8,
    "batch_size": 128,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "patience": 3,
    "min_delta": 0.001,
    "lr_decay_factor": 0.5,
    "lr_patience": 1,
    "gradient_clip_norm": 1.0,
}

_loader_torch = globals().get("torch", None)
TRANSFORMER_LOADER_CONFIG = {
    "num_workers": 0,
    "pin_memory": bool(_loader_torch is not None and _loader_torch.cuda.is_available()),
    "persistent_workers": False,
    "prefetch_factor": 2,
}

display(pd.Series(TRANSFORMER_CONFIG, name="value").rename_axis("transformer_parameter").to_frame())
display(pd.Series(TRANSFORMER_LOADER_CONFIG, name="value").rename_axis("loader_parameter").to_frame())


### Transformer Step 1: Prepare Windowed Sequence Data

The transformer uses the same fixed split as the CNN, but it keeps the sequence in its original time-major layout.

In other words:

- CNN sees `[batch, channels, time]`,
- transformer sees `[batch, time, features]`.

Just like the CNN, the transformer here uses windows of the selected measurement columns rather than the full Random Forest feature table. So it gets temporal context directly from the sequence, not from handcrafted calendar or delta features.

The `output_mode` toggle also matters here:

- `"window"` trains one prediction for the whole window,
- `"per_timestep"` trains one prediction at every position in the window.

That makes this step a nice comparison point between two sequence models trained on the same scalar QC problem.


### Transformer Step 0: Release Neural Model Memory And Load PyTorch

If you ran the CNN section just before this one, the notebook may still be holding large tensors, `DataLoader` objects, and model state.

This cleanup cell frees those objects before the transformer starts. It also imports PyTorch here so the transformer section can be run even if you skipped the CNN cells.


In [ ]:
import gc

try:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset

    PYTORCH_READY = True
except ImportError as exc:
    PYTORCH_READY = False
    print("PyTorch is not installed in this environment.")
    raise SystemExit(exc)

# Clear large sequence-model objects left behind by earlier sections before
# building transformer windows. This is especially important on shared
# Jupyter kernels where CPU RAM is limited.
released_names = []
for name in [
    "cnn_bundle",
    "sequence_split_frames_materialized",
    "train_dataset",
    "valid_dataset",
    "test_dataset",
    "train_loader",
    "valid_loader",
    "test_loader",
    "train_targets_tensor",
    "valid_targets_tensor",
    "test_targets_tensor",
    "X_train",
    "X_valid",
    "X_test",
    "y_train",
    "y_valid",
    "y_test",
    "model",
    "optimizer",
    "scheduler",
    "loss_fn",
    "best_state",
    "history",
]:
    if name in globals():
        del globals()[name]
        released_names.append(name)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(
    {
        "released_objects": released_names,
        "released_count": len(released_names),
        "torch": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
    }
)


In [ ]:
if not globals().get("PYTORCH_READY", False):
    TRANSFORMER_RUN = False
    print("Transformer training cell skipped because PyTorch is not available in the current notebook state.")
else:
    sequence_split_frames_materialized = materialize_reviewed_split_frames(
        ML_STATE['selected_paths'],
        {"train": ML_STATE['train_df'], "validation": ML_STATE['valid_df'], "test": ML_STATE['test_df']},
        columns=ML_STATE['ROW_USE_COLUMNS'],
        target_flag=ML_STATE['TARGET_FLAG'],
        task_mode=ML_STATE['task_mode'],
        good_labels=ML_STATE['GOOD_LABELS'],
        issue_labels=ML_STATE['ISSUE_LABELS'],
    )
    transformer_window_size = TRANSFORMER_CONFIG["window_size"]
    TRANSFORMER_OUTPUT_MODE = TRANSFORMER_CONFIG["output_mode"]
    transformer_bundle = build_sequence_split_bundle_from_frames(
        sequence_split_frames_materialized,
        measurement_columns=ML_STATE['measurement_columns'],
        target_column="model_target",
        task_mode=ML_STATE['task_mode'],
        output_mode=TRANSFORMER_OUTPUT_MODE,
        window_size=transformer_window_size,
        label_reduction=TRANSFORMER_CONFIG["label_reduction"],
    )
    transformer_class_labels = transformer_bundle.class_labels

    if ML_STATE['task_mode'] == "multiclass" and len(transformer_class_labels) < 2:
        TRANSFORMER_RUN = False
        print(
            {
                "output_mode": TRANSFORMER_OUTPUT_MODE,
                "windows_total": int(len(transformer_bundle.X_train) + len(transformer_bundle.X_valid) + len(transformer_bundle.X_test)),
                "active_labels": transformer_class_labels,
                "skip_reason": "Need at least two active classes to train the transformer in multiclass mode.",
            }
        )
    elif len(transformer_bundle.X_train) == 0 or len(transformer_bundle.X_valid) == 0 or len(transformer_bundle.X_test) == 0:
        TRANSFORMER_RUN = False
        print(
            {
                "output_mode": TRANSFORMER_OUTPUT_MODE,
                "split_strategy": ML_STATE['FIXED_SPLIT_STRATEGY'],
                "train_subset_strategy": ML_STATE['TRAIN_SUBSET_STRATEGY'],
                "skip_reason": "At least one split produced no full windows for this configuration.",
            }
        )
    else:
        X_train = transformer_bundle.X_train
        X_valid = transformer_bundle.X_valid
        X_test = transformer_bundle.X_test
        y_train = transformer_bundle.y_train
        y_valid = transformer_bundle.y_valid
        y_test = transformer_bundle.y_test

        feature_mean = X_train.mean(axis=(0, 1), keepdims=True)
        feature_std = X_train.std(axis=(0, 1), keepdims=True) + 1e-6
        X_train = ((X_train - feature_mean) / feature_std).astype(np.float32, copy=False)
        X_valid = ((X_valid - feature_mean) / feature_std).astype(np.float32, copy=False)
        X_test = ((X_test - feature_mean) / feature_std).astype(np.float32, copy=False)

        del transformer_bundle, sequence_split_frames_materialized
        import gc
        gc.collect()

        TRANSFORMER_RUN = True
        print(
            {
                "output_mode": TRANSFORMER_OUTPUT_MODE,
                "split_strategy": ML_STATE['FIXED_SPLIT_STRATEGY'],
                "train_subset_strategy": ML_STATE['TRAIN_SUBSET_STRATEGY'],
                "fixed_split_block_rows": ML_STATE['FIXED_SPLIT_BLOCK_ROWS'] if ML_STATE['FIXED_SPLIT_STRATEGY'] in {"interleaved_blocks", "episode_aware"} else None,
                "windows_total": int(len(X_train) + len(X_valid) + len(X_test)),
                "train_windows": int(len(X_train)),
                "validation_windows": int(len(X_valid)),
                "test_windows": int(len(X_test)),
                "window_size": int(transformer_window_size),
                "features_per_step": int(len(ML_STATE['measurement_columns'])),
            }
        )


### Transformer Step 2: Build Attention Blocks, Loss, and DataLoaders

The transformer needs a few ingredients that are specific to attention models:

- an input projection that maps raw sensor features into `d_model`,
- a positional embedding so the model knows where each time step sits in the window,
- stacked encoder blocks with multi-head attention,
- and, in window mode, a pooling rule that turns the whole window into one classification vector.

The output toggle changes the last step:

- in `"window"` mode, we pool the encoded sequence and classify once,
- in `"per_timestep"` mode, we classify every encoded position directly.

Everything else should feel familiar from the CNN section: loss, optimiser, scheduler, and `DataLoader` setup.


In [ ]:
if not globals().get("TRANSFORMER_RUN", False):
    print("Transformer model setup skipped.")
else:
    torch.manual_seed(ML_STATE['SEED'])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cuda":
        torch.backends.cudnn.benchmark = True

    class TinyQCTransformer(nn.Module):
        def __init__(self, input_dim: int, output_dim: int) -> None:
            super().__init__()
            self.output_dim = output_dim
            self.input_projection = nn.Linear(input_dim, TRANSFORMER_CONFIG["d_model"])
            self.position_embedding = nn.Parameter(
                torch.zeros(1, TRANSFORMER_CONFIG["window_size"], TRANSFORMER_CONFIG["d_model"])
            )
            encoder_layer = nn.TransformerEncoderLayer(
                d_model=TRANSFORMER_CONFIG["d_model"],
                nhead=TRANSFORMER_CONFIG["nhead"],
                dim_feedforward=TRANSFORMER_CONFIG["dim_feedforward"],
                dropout=TRANSFORMER_CONFIG["dropout"],
                activation="gelu",
                batch_first=True,
            )
            self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=TRANSFORMER_CONFIG["num_layers"])
            self.norm = nn.LayerNorm(TRANSFORMER_CONFIG["d_model"])
            self.head = nn.Linear(TRANSFORMER_CONFIG["d_model"], output_dim)

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            x = self.input_projection(x)
            x = x + self.position_embedding[:, : x.size(1)]
            x = self.encoder(x)
            if TRANSFORMER_OUTPUT_MODE == "window":
                if TRANSFORMER_CONFIG["pooling"] == "last":
                    pooled = x[:, -1, :]
                else:
                    pooled = x.mean(dim=1)
                pooled = self.norm(pooled)
                return self.head(pooled)
            x = self.norm(x)
            logits = self.head(x)
            if self.output_dim == 1:
                return logits.squeeze(-1)
            return logits

    if ML_STATE['task_mode'] == "multiclass":
        train_targets_tensor = torch.from_numpy(y_train).long()
        valid_targets_tensor = torch.from_numpy(y_valid).long()
        test_targets_tensor = torch.from_numpy(y_test).long()
        class_counts = np.bincount(y_train.reshape(-1), minlength=len(transformer_class_labels)).clip(min=1)
        class_weights = y_train.size / (len(class_counts) * class_counts)
        loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=device))
        output_dim = len(transformer_class_labels)
    else:
        train_targets_tensor = torch.from_numpy(y_train.astype(np.float32))
        valid_targets_tensor = torch.from_numpy(y_valid.astype(np.float32))
        test_targets_tensor = torch.from_numpy(y_test.astype(np.float32))
        positive_count = max(float(y_train.sum()), 1.0)
        negative_count = max(float(y_train.size - y_train.sum()), 1.0)
        pos_weight = torch.tensor([negative_count / positive_count], dtype=torch.float32, device=device)
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        output_dim = 1

    train_dataset = TensorDataset(torch.from_numpy(np.ascontiguousarray(X_train)), train_targets_tensor)
    valid_dataset = TensorDataset(torch.from_numpy(np.ascontiguousarray(X_valid)), valid_targets_tensor)
    test_dataset = TensorDataset(torch.from_numpy(np.ascontiguousarray(X_test)), test_targets_tensor)

    loader_kwargs = {}
    num_workers = int(TRANSFORMER_LOADER_CONFIG["num_workers"])
    if num_workers > 0:
        loader_kwargs["num_workers"] = num_workers
        loader_kwargs["persistent_workers"] = bool(TRANSFORMER_LOADER_CONFIG["persistent_workers"])
        loader_kwargs["prefetch_factor"] = int(TRANSFORMER_LOADER_CONFIG["prefetch_factor"])
    if TRANSFORMER_LOADER_CONFIG["pin_memory"] and device.type == "cuda":
        loader_kwargs["pin_memory"] = True
    use_non_blocking = bool(loader_kwargs.get("pin_memory", False) and device.type == "cuda")

    del X_train, X_valid, X_test, y_train, y_valid, y_test
    del train_targets_tensor, valid_targets_tensor, test_targets_tensor
    import gc
    gc.collect()

    train_loader = DataLoader(train_dataset, batch_size=TRANSFORMER_CONFIG["batch_size"], shuffle=True, **loader_kwargs)
    valid_loader = DataLoader(valid_dataset, batch_size=TRANSFORMER_CONFIG["batch_size"], shuffle=False, **loader_kwargs)
    test_loader = DataLoader(test_dataset, batch_size=TRANSFORMER_CONFIG["batch_size"], shuffle=False, **loader_kwargs)

    model = TinyQCTransformer(input_dim=len(ML_STATE['measurement_columns']), output_dim=output_dim).to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=TRANSFORMER_CONFIG["learning_rate"],
        weight_decay=TRANSFORMER_CONFIG["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=TRANSFORMER_CONFIG["lr_decay_factor"],
        patience=TRANSFORMER_CONFIG["lr_patience"],
    )

    best_metric = -np.inf
    best_epoch = 0
    patience_counter = 0
    history = []
    best_state = None

    def run_epoch(loader, training: bool) -> tuple[float, np.ndarray, np.ndarray]:
        model.train(training)
        total_loss = 0.0
        predictions = []
        targets = []
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device, non_blocking=use_non_blocking)
            batch_y = batch_y.to(device, non_blocking=use_non_blocking)
            with torch.set_grad_enabled(training):
                logits = model(batch_x)
                if ML_STATE['task_mode'] == "multiclass":
                    if TRANSFORMER_OUTPUT_MODE == "window":
                        loss = loss_fn(logits, batch_y)
                        batch_predictions = logits.argmax(dim=1)
                    else:
                        loss = loss_fn(logits.permute(0, 2, 1), batch_y)
                        batch_predictions = logits.argmax(dim=-1)
                else:
                    loss = loss_fn(logits, batch_y)
                    batch_predictions = (torch.sigmoid(logits) >= 0.5).long()
                if training:
                    optimizer.zero_grad(set_to_none=True)
                    loss.backward()
                    if TRANSFORMER_CONFIG["gradient_clip_norm"]:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), TRANSFORMER_CONFIG["gradient_clip_norm"])
                    optimizer.step()
            total_loss += float(loss.item()) * len(batch_x)
            predictions.append(batch_predictions.detach().cpu().reshape(-1).numpy())
            targets.append(batch_y.detach().cpu().reshape(-1).numpy())
        predictions_array = np.concatenate(predictions) if predictions else np.array([])
        targets_array = np.concatenate(targets) if targets else np.array([])
        average_loss = total_loss / max(len(loader.dataset), 1)
        return average_loss, predictions_array, targets_array

    print(
        {
            "device": str(device),
            "output_mode": TRANSFORMER_OUTPUT_MODE,
            "train_batches": len(train_loader),
            "validation_batches": len(valid_loader),
            "test_batches": len(test_loader),
            "loader_config": TRANSFORMER_LOADER_CONFIG,
        }
    )


### Transformer Step 3: Train with Attention-Based Sequence Modelling

The training loop is almost the same as the CNN loop, which is useful pedagogically: once the data pipeline and validation workflow are clear, we can swap in a different model family without changing the whole experimental process.

Just like the CNN section, the reported validation F1 is per window in `"window"` mode and per timestamp in `"per_timestep"` mode.


In [ ]:
if not globals().get("TRANSFORMER_RUN", False):
    print("Transformer fit skipped.")
else:
    for epoch in range(1, TRANSFORMER_CONFIG["epochs"] + 1):
        train_loss, _, _ = run_epoch(train_loader, training=True)
        valid_loss, valid_preds, valid_targets = run_epoch(valid_loader, training=False)
        valid_f1 = f1_score(valid_targets, valid_preds, average=report_average(ML_STATE['task_mode']), zero_division=0)
        scheduler.step(valid_f1)
        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "valid_loss": valid_loss,
                "valid_f1": valid_f1,
                "learning_rate": optimizer.param_groups[0]["lr"],
            }
        )
        print(
            f"Epoch {epoch:>2} | train_loss={train_loss:.4f} | valid_loss={valid_loss:.4f} | valid_f1={valid_f1:.4f}"
        )

        if valid_f1 > best_metric + TRANSFORMER_CONFIG["min_delta"]:
            best_metric = valid_f1
            best_epoch = epoch
            patience_counter = 0
            best_state = copy.deepcopy(model.state_dict())
            torch.save(
                {
                    "model_state_dict": best_state,
                    "task_mode": ML_STATE['task_mode'],
                    "class_labels": transformer_class_labels,
                    "feature_columns": ML_STATE['measurement_columns'],
                    "window_size": transformer_window_size,
                    "output_mode": TRANSFORMER_OUTPUT_MODE,
                    "config": {**TRANSFORMER_CONFIG, **TRANSFORMER_LOADER_CONFIG},
                },
                ML_STATE["TRANSFORMER_MODEL_PATH"],
            )
        else:
            patience_counter += 1
            if patience_counter >= TRANSFORMER_CONFIG["patience"]:
                print(f"Early stopping triggered at epoch {epoch}")
                break


### Transformer Step 4: Evaluate the Best Attention Model

Just like the CNN section, we finish by restoring the best validation checkpoint and scoring the held-out test split exactly once.


In [ ]:
if not globals().get("TRANSFORMER_RUN", False):
    print("Transformer evaluation skipped.")
else:
    if best_state is None:
        raise RuntimeError("Transformer training did not produce a valid checkpoint")

    model.load_state_dict(best_state)
    test_loss, test_preds, test_targets = run_epoch(test_loader, training=False)

    print(pd.DataFrame(history).tail())
    print(
        {
            "output_mode": TRANSFORMER_CONFIG["output_mode"],
            "windows": len(train_dataset) + len(valid_dataset) + len(test_dataset),
            "device": str(device),
            "best_validation_f1": float(best_metric),
            "best_epoch": best_epoch,
            "test_loss": float(test_loss),
            "saved_model": str(ML_STATE["TRANSFORMER_MODEL_PATH"]),
        }
    )

    if ML_STATE['task_mode'] == "multiclass":
        report_labels = list(range(len(transformer_class_labels)))
        report_names = [str(label) for label in transformer_class_labels]
    else:
        report_labels = [0, 1]
        report_names = ["0", "1"]

    print(
        classification_report(
            test_targets,
            test_preds,
            labels=report_labels,
            target_names=report_names,
            zero_division=0,
        )
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    history_frame = pd.DataFrame(history)
    axes[0].plot(history_frame["epoch"], history_frame["train_loss"], label="train")
    axes[0].plot(history_frame["epoch"], history_frame["valid_loss"], label="validation")
    axes[0].set_title("Transformer loss history")
    axes[0].legend()
    ConfusionMatrixDisplay.from_predictions(
        test_targets,
        test_preds,
        labels=report_labels,
        display_labels=report_names,
        normalize="true",
        xticks_rotation=45,
        ax=axes[1],
    )
    axes[1].set_title("Transformer test confusion matrix")
    plt.tight_layout()
    plt.show()


### Date-Range Demo: See Transformer Predictions on a Specific Interval

This demo follows the same `output_mode` toggle as the training section:

- `"window"` shows one true/predicted label span per window,
- `"per_timestep"` shows one true/predicted label span per timestamp.


In [ ]:
TRANSFORMER_RANGE_START = None
TRANSFORMER_RANGE_END = None
TRANSFORMER_AUTO_SELECT_TEST_RANGE = True
TRANSFORMER_MAX_POINTS_TO_PLOT = 1024

print(
    {
        "TRANSFORMER_RANGE_START": TRANSFORMER_RANGE_START,
        "TRANSFORMER_RANGE_END": TRANSFORMER_RANGE_END,
        "TRANSFORMER_AUTO_SELECT_TEST_RANGE": TRANSFORMER_AUTO_SELECT_TEST_RANGE,
        "TRANSFORMER_MAX_POINTS_TO_PLOT": TRANSFORMER_MAX_POINTS_TO_PLOT,
    }
)


In [ ]:
if not globals().get("TRANSFORMER_RUN", False):
    print("Transformer date-range demo skipped because the transformer was not trained in this run.")
else:
    transformer_range_selection = select_time_range(
        ML_STATE['test_df'],
        time_column="Time UTC",
        label_column=ML_STATE['TARGET_FLAG'],
        start=TRANSFORMER_RANGE_START,
        end=TRANSFORMER_RANGE_END,
        auto_select=TRANSFORMER_AUTO_SELECT_TEST_RANGE,
        max_points=TRANSFORMER_MAX_POINTS_TO_PLOT,
    )
    transformer_interval_metrics = None

    transformer_interval_rows = load_rows_for_time_range(
        ML_STATE['metadata'],
        row_cache_dir=Path(ML_STATE["ROW_CACHE_DIR"]),
        start=transformer_range_selection["start"],
        end=transformer_range_selection["end"],
        columns=ML_STATE['ROW_USE_COLUMNS'],
    )

    if transformer_interval_rows.empty:
        print("No row-level data was found in the requested transformer range.")
    else:
        transformer_interval_model_df, _, _ = build_model_frame(
            transformer_interval_rows,
            target_flag=ML_STATE['TARGET_FLAG'],
            measurement_columns=ML_STATE['measurement_columns'],
            task_mode=ML_STATE['task_mode'],
            model_row_limit=None,
        )
        transformer_interval_model_df = transformer_interval_model_df[
            (transformer_interval_model_df["Time UTC"] >= transformer_range_selection["start"])
            & (transformer_interval_model_df["Time UTC"] <= transformer_range_selection["end"])
        ].reset_index(drop=True)
        transformer_interval_origin = infer_interval_origin(
            transformer_range_selection["start"],
            transformer_range_selection["end"],
            {"train": ML_STATE['train_df'], "validation": ML_STATE['valid_df'], "test": ML_STATE['test_df']},
        )
        transformer_plot_palette = DEFAULT_FLAG_PALETTE if ML_STATE['task_mode'] == "multiclass" else {0: "#1f77b4", 1: "#d62728"}

        if TRANSFORMER_CONFIG["output_mode"] == "window":
            transformer_interval_bundle = build_window_classification_interval_data(
                transformer_interval_model_df,
                feature_columns=ML_STATE['measurement_columns'],
                target_column="model_target",
                task_mode=ML_STATE['task_mode'],
                window_size=TRANSFORMER_CONFIG["window_size"],
                label_reduction=TRANSFORMER_CONFIG["label_reduction"],
            )

            if transformer_interval_bundle["window_frame"].empty:
                print("The selected transformer range is shorter than one full window after preprocessing, so the window-level demo is skipped.")
            else:
                transformer_predicted_labels = predict_transformer_window_model(
                    model,
                    transformer_interval_bundle["raw_sequences"],
                    task_mode=ML_STATE['task_mode'],
                    class_labels=transformer_class_labels,
                    device=str(device),
                    feature_mean=feature_mean,
                    feature_std=feature_std,
                    batch_size=TRANSFORMER_CONFIG["batch_size"],
                )
                transformer_window_frame = transformer_interval_bundle["window_frame"].copy()
                transformer_window_frame["predicted_label"] = transformer_predicted_labels
                transformer_interval_metrics = compute_interval_classification_metrics(
                    transformer_window_frame["true_label"],
                    transformer_window_frame["predicted_label"],
                    labels=transformer_class_labels,
                    average=report_average(ML_STATE['task_mode']),
                    target_names=[str(label) for label in transformer_class_labels],
                )
                transformer_true_intervals = merge_adjacent_intervals(
                    transformer_window_frame.rename(columns={"window_start": "start", "window_end": "end", "true_label": "label"})[
                        ["start", "end", "label"]
                    ]
                )
                transformer_pred_intervals = merge_adjacent_intervals(
                    transformer_window_frame.rename(columns={"window_start": "start", "window_end": "end", "predicted_label": "label"})[
                        ["start", "end", "label"]
                    ]
                )

                print(
                    {
                        "output_mode": TRANSFORMER_CONFIG["output_mode"],
                        "selection_mode": transformer_range_selection["selection_mode"],
                        "selected_priority_flag": transformer_range_selection["selected_label"],
                        "interval_origin": transformer_interval_origin,
                        "range_start": transformer_range_selection["start"].isoformat(),
                        "range_end": transformer_range_selection["end"].isoformat(),
                        "window_count_in_interval": int(len(transformer_window_frame)),
                        "interval_f1": transformer_interval_metrics["f1"],
                    }
                )
                print(transformer_interval_metrics["report_text"])
                display(
                    pd.DataFrame(
                        {
                            "true_count": transformer_window_frame["true_label"].value_counts().sort_index(),
                            "predicted_count": transformer_window_frame["predicted_label"].value_counts().sort_index(),
                        }
                    ).fillna(0).astype(int)
                )

                transformer_demo_figure = plot_time_series_with_bands(
                    transformer_interval_model_df,
                    band_specs=[
                        {"title": f"True window {ML_STATE['FLAG_EXAMPLE_DISPLAY_NAME']}", "intervals": transformer_true_intervals, "palette": transformer_plot_palette},
                        {"title": f"Transformer {ML_STATE['FLAG_EXAMPLE_DISPLAY_NAME']} prediction", "intervals": transformer_pred_intervals, "palette": transformer_plot_palette},
                    ],
                    measurement_column=ML_STATE['PLOT_MEASUREMENT_COLUMN'],
                    secondary_column=ML_STATE['PLOT_SECONDARY_COLUMN'],
                    max_points=TRANSFORMER_MAX_POINTS_TO_PLOT,
                    title="Transformer window predictions on a selected time range",
                    label_meanings=ML_STATE['FLAG_EXAMPLE_LABEL_MEANINGS'],
                    legend_title=ML_STATE['FLAG_EXAMPLE_DISPLAY_NAME'],
                )
                plt.show()
        else:
            transformer_interval_bundle = build_sequence_label_interval_data(
                transformer_interval_model_df,
                feature_columns=ML_STATE['measurement_columns'],
                target_column="model_target",
                window_size=TRANSFORMER_CONFIG["window_size"],
            )

            if len(transformer_interval_bundle["raw_sequences"]) == 0:
                print("The selected transformer range is shorter than one full window after preprocessing, so the per-timestep demo is skipped.")
            else:
                transformer_predicted_labels = predict_transformer_sequence_model(
                    model,
                    transformer_interval_bundle["raw_sequences"],
                    task_mode=ML_STATE['task_mode'],
                    class_labels=transformer_class_labels,
                    device=str(device),
                    feature_mean=feature_mean,
                    feature_std=feature_std,
                    batch_size=TRANSFORMER_CONFIG["batch_size"],
                )
                transformer_point_frame = pd.DataFrame(
                    {
                        "Time UTC": pd.to_datetime(transformer_interval_bundle["raw_times"].reshape(-1), utc=True),
                        "true_label": transformer_interval_bundle["raw_targets"].reshape(-1).astype(int),
                        "predicted_label": transformer_predicted_labels.reshape(-1).astype(int),
                    }
                )
                transformer_interval_metrics = compute_interval_classification_metrics(
                    transformer_point_frame["true_label"],
                    transformer_point_frame["predicted_label"],
                    labels=transformer_class_labels,
                    average=report_average(ML_STATE['task_mode']),
                    target_names=[str(label) for label in transformer_class_labels],
                )
                transformer_true_intervals = merge_adjacent_intervals(
                    build_labeled_intervals(transformer_point_frame, time_column="Time UTC", label_column="true_label")
                )
                transformer_pred_intervals = merge_adjacent_intervals(
                    build_labeled_intervals(transformer_point_frame, time_column="Time UTC", label_column="predicted_label")
                )

                print(
                    {
                        "output_mode": TRANSFORMER_CONFIG["output_mode"],
                        "selection_mode": transformer_range_selection["selection_mode"],
                        "selected_priority_flag": transformer_range_selection["selected_label"],
                        "interval_origin": transformer_interval_origin,
                        "range_start": transformer_range_selection["start"].isoformat(),
                        "range_end": transformer_range_selection["end"].isoformat(),
                        "point_count_in_interval": int(len(transformer_point_frame)),
                        "interval_f1": transformer_interval_metrics["f1"],
                    }
                )
                print(transformer_interval_metrics["report_text"])
                display(
                    pd.DataFrame(
                        {
                            "true_count": transformer_point_frame["true_label"].value_counts().sort_index(),
                            "predicted_count": transformer_point_frame["predicted_label"].value_counts().sort_index(),
                        }
                    ).fillna(0).astype(int)
                )

                transformer_demo_figure = plot_time_series_with_bands(
                    transformer_interval_model_df,
                    band_specs=[
                        {"title": f"True per-timestep {ML_STATE['FLAG_EXAMPLE_DISPLAY_NAME']}", "intervals": transformer_true_intervals, "palette": transformer_plot_palette},
                        {"title": f"Transformer per-timestep {ML_STATE['FLAG_EXAMPLE_DISPLAY_NAME']} prediction", "intervals": transformer_pred_intervals, "palette": transformer_plot_palette},
                    ],
                    measurement_column=ML_STATE['PLOT_MEASUREMENT_COLUMN'],
                    secondary_column=ML_STATE['PLOT_SECONDARY_COLUMN'],
                    max_points=TRANSFORMER_MAX_POINTS_TO_PLOT,
                    title="Transformer per-timestep predictions on a selected time range",
                    label_meanings=ML_STATE['FLAG_EXAMPLE_LABEL_MEANINGS'],
                    legend_title=ML_STATE['FLAG_EXAMPLE_DISPLAY_NAME'],
                )
                plt.show()

        if transformer_interval_metrics is not None:
            transformer_cm_fig, transformer_cm_ax = plt.subplots(figsize=(5, 4))
            ConfusionMatrixDisplay(
                confusion_matrix=transformer_interval_metrics["confusion_matrix"],
                display_labels=transformer_interval_metrics["display_labels"],
            ).plot(ax=transformer_cm_ax, cmap="Blues", colorbar=False)
            transformer_cm_ax.set_title("Transformer confusion matrix on the selected range")
            plt.tight_layout()
            plt.show()


### Reflection Questions: Transformer

1. Does the selected interval contain longer-context behaviour that you would expect a transformer to use better than the CNN?
2. If the transformer is not clearly helping here, do you think the issue is window length, data volume, or model size?
3. Would you change pooling, positional information, or the target definition first if you wanted a better interval-level result?

Add your own notes or follow-up questions here:

- 
- 
- 


## Part 5 — Reflection and Next Steps


### Wrap-Up, Transformer Note, And Questions To Explore

We now have three very different model families in one notebook:

- a tabular tree ensemble,
- a local-pattern sequence model,
- and a small attention-based sequence model.

That is a useful comparison because each one sees the data a little differently.

Good next questions to explore:

1. Are there other features that might help the Random Forest?
Hint: think about lag features, rolling means, rolling standard deviations, slopes, or relationships between sensors.

2. Are there ways to improve class `3`, `4`, or `9` performance specifically?
Hint: compare the class distribution to the confusion matrix and ask which classes are hardest to distinguish.

3. Does the target itself need to change?
Hint: sometimes a binary `good` vs `issue` target, or a collapsed set of flags, is closer to the operational decision. For conductivity or turbidity, that might mean merging `3` and `4`. For oxygen, it can make more sense to ask whether `1/2` belong together and whether `3/4` belong together, rather than reusing the CTD collapse blindly.

4. What parts of the CNN are worth experimenting with?
Hint: try changing the window length, number of channels, dropout, or the way we reduce row labels into one window label.

5. Could we use even more context from time?
Hint: Random Forest only sees engineered features. CNN sees short windows. Transformer sees relationships across an entire window. There are still other options too, such as TCNs, GRUs/LSTMs, or hierarchical windows.
